# Image Captioning Model Testing

- This notebook containes a variety of Image Captioning Models tested on inference speed & caption quality to determine which Model is appropriate for captioning Million scale Images.

### Dataloader used to efficiently load image Batches for inference

In [1]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T
import numpy as np
import torch
import os

# ---------------- Dataset ----------------
class ImageDataset(Dataset):
    def __init__(self, image_dir: str, extensions=(".jpg", ".jpeg", ".png"), as_tensor=False, transform=None):
        self.image_paths = [
            os.path.join(image_dir, f)
            for f in os.listdir(image_dir)
            if f.lower().endswith(extensions)
        ]
        self.as_tensor = as_tensor
        self.transform = transform or (T.ToTensor() if as_tensor else None)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = self.image_paths[idx]
        img = Image.open(path).convert("RGB")

        if self.as_tensor and self.transform:
            img = self.transform(img)

        return img, path

# ---------------- Collate Function ----------------
def collate_images(batch):
    imgs, paths = zip(*batch)
    return list(imgs), list(paths)

# ---------------- Prefetching Loader ----------------
class PrefetchLoader:
    """Wrap a DataLoader to prefetch the next batch to GPU asynchronously."""
    def __init__(self, loader, device, as_tensor=False):
        self.loader = iter(loader)
        self.device = device
        self.as_tensor = as_tensor
        self.stream = torch.cuda.Stream() if device == "cuda" else None
        self.next_batch = None
        self._prefetch()

    def _prefetch(self):
        try:
            imgs, paths = next(self.loader)
        except StopIteration:
            self.next_batch = None
            return

        # Prefetching logic
        if self.device == "cuda":
            with torch.cuda.stream(self.stream):
                if self.as_tensor:
                    # Move tensors to GPU asynchronously
                    imgs = [img.to(self.device, non_blocking=True) for img in imgs]
                # else: keep PIL images on CPU (LLaVA/CLIP-style models expect CPU PILs)
        else:
            if self.as_tensor:
                imgs = [img.to(self.device) for img in imgs]

        self.next_batch = (imgs, paths)

    def __iter__(self):
        return self

    def __next__(self):
        if self.next_batch is None:
            raise StopIteration
        batch = self.next_batch
        self._prefetch()
        return batch

# ---------------- DataLoader setup ----------------
def get_image_loader(image_dir: str, batch_size=4, num_workers=4, device='cpu', as_tensor=False):
    dataset = ImageDataset(image_dir, as_tensor=as_tensor)
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        num_workers=num_workers,
        shuffle=False,
        collate_fn=collate_images,
        pin_memory=True
    )
    prefetch_loader = PrefetchLoader(loader, device=device, as_tensor=as_tensor)
    return prefetch_loader, len(dataset)

## Model 1 - llava-hf/llava-1.5-7b-hf

- This is the model used in the ["Understanding Bias in Large-Scale Visual Datasets"](https://arxiv.org/pdf/2412.01876v1) paper, using 4bit quantization as per the  [github repo](https://github.com/boyazeng/understand_bias/blob/main/transformations/caption/transform.py).

- **NOT VIABLE**: Model takes 27.38 seconds to process 25 images using an batch_size=2 (larger batch sizes slow down inference due to model scale) even when scaled down with the 4bit quantization

In [2]:
import torch
from PIL import Image
import time
from transformers import LlavaProcessor, LlavaForConditionalGeneration, BitsAndBytesConfig
import matplotlib.pyplot as plt
import torchvision.transforms.functional as F

# Load the dataset
image_dir = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\images"
prefetch_loader, dataset_size = get_image_loader(image_dir, batch_size=2, num_workers=0, device="cuda")

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

model_id = "llava-hf/llava-1.5-7b-hf"
processor = LlavaProcessor.from_pretrained(model_id, use_fast=True)
model = LlavaForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="cuda"
)

def generate_captions_batch(images, paths, caption_type="short"):
    if caption_type == "short":
        prompt = "USER: <image>\nDescribe this image in one sentence.\nASSISTANT:"
        max_tokens = 50
    else:
        prompt = "USER: <image>\nDescribe this image in one paragraph.\nASSISTANT:"
        max_tokens = 150

    # Combine prompts with each image
    prompts = [prompt] * len(images)

    # Prepare inputs — LLaVA can handle batched images
    inputs = processor(
        images=images,
        text=prompts,
        return_tensors="pt",
        padding=True
    ).to(model.device)

    start_time = time.time()
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_tokens)
    print(f"Time taken: {time.time() - start_time}")

    # Decode each caption separately
    captions = []
    for output in outputs:
        text = processor.decode(output, skip_special_tokens=True)
        text = text.split("ASSISTANT:")[-1].strip()
        captions.append(text)

    return list(zip(paths, captions))

all_results = []

for imgs, paths in prefetch_loader:
    # Generate captions for the whole batch
    batch_results = generate_captions_batch(imgs, paths, caption_type="short")
        
    all_results.extend(batch_results)

# Example output
for path, caption in all_results:
    print(f"{os.path.basename(path)}: {caption}")

Skipping import of cpp extensions due to incompatible torch version 2.6.0+cu124 for torchao version 0.14.0         Please see GitHub issue #2919 for more info


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Time taken: 2.6840641498565674
Time taken: 2.4764938354492188
Time taken: 1.8138954639434814
Time taken: 1.8158586025238037
Time taken: 2.3826143741607666
Time taken: 1.916088342666626
Time taken: 2.21797776222229
Time taken: 1.7319316864013672
Time taken: 2.200808525085449
Time taken: 2.195845365524292
Time taken: 2.191230297088623
Time taken: 2.5566318035125732
Time taken: 1.196281909942627
a car_1 - Copy.png: A car is driving down a street with a large building in the background.
a car_1.png: A silver car is driving down a busy street.
a car_2 - Copy.png: A silver car with red and yellow stripes is parked on the side of the road.
a car_2.png: A silver car with red and yellow stripes is parked on the side of the road.
a car_3.png: A group of men are working on a car.
a car_4.png: A green car is parked in front of a building.
a car_5.png: A red and blue car is driving down a street.
a car_6.png: A white car is parked in a grassy field.
a car_7.png: A blue car is parked next to a yello

## Model 2 - tinyllava/TinyLLaVA-Phi-2-SigLIP-3.1B

- This is a Tiny version of the llava model available on [hugging face](https://huggingface.co/tinyllava/TinyLLaVA-Phi-2-SigLIP-3.1B) supposedly achieving better overall performance against existing 7B models such as LLaVA-1.5 and Qwen-VL.

- This variant does not appear to support 4bit quantization making it worse off than the **llava-hf/llava-1.5-7b-hf** model.

- **NOT VIABLE**: Model takes 3 minutes to process 25 images using an batch_size=2 (larger batch sizes slow down inference due to model scale).

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import os, time
from tqdm import tqdm
from generate_model import generate  # official TinyLLaVA helper

# -------------------------------
# 1. Configuration
# -------------------------------
image_dir = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\images"
batch_size = 2
caption_type = "short"  # "short" = 1 sentence, "long" = paragraph
device = "cuda" if torch.cuda.is_available() else "cpu"

# -------------------------------
# 2. Load TinyLLaVA model
# -------------------------------
hf_path = 'tinyllava/TinyLLaVA-Phi-2-SigLIP-3.1B'

model = AutoModelForCausalLM.from_pretrained(
    hf_path,
    trust_remote_code=True,
    attn_implementation="eager",
    dtype=torch.float16,
    device_map="auto"
).eval()

config = model.config
tokenizer = AutoTokenizer.from_pretrained(
    hf_path,
    use_fast=True,
    model_max_length=config.tokenizer_model_max_length,
    padding_side=config.tokenizer_padding_side
)

# -------------------------------
# 3. Helper to load image paths
# -------------------------------
def load_image_paths(directory):
    exts = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
    return [os.path.join(directory, f) for f in os.listdir(directory) if f.lower().endswith(exts)]

image_paths = load_image_paths(image_dir)
print(f"Found {len(image_paths)} images")

# -------------------------------
# 4. Caption generation
# -------------------------------
def generate_captions_batch(paths, caption_type="short"):
    if caption_type == "short":
        prompt_text = "Describe this image in one sentence."
    else:
        prompt_text = "Describe this image in one paragraph."

    captions = []

    for path in paths:
        start_time = time.time()
        # TinyLLaVA generate() accepts local image paths
        output_text, generation_time = generate(
            prompt=prompt_text,
            image=path,
            model=model,
            tokenizer=tokenizer
        )
        print(f"Processed Batch in {generation_time:.2f}s")
        captions.append((path, output_text))

    return captions

# -------------------------------
# 5. Process images in batches
# -------------------------------
all_results = []

for i in tqdm(range(0, len(image_paths), batch_size), desc="Processing batches"):
    batch_paths = image_paths[i:i + batch_size]
    batch_results = generate_captions_batch(batch_paths, caption_type=caption_type)
    all_results.extend(batch_results)

# -------------------------------
# 6. Output results
# -------------------------------
print("\n=== Captions ===")
for path, caption in all_results:
    print(f"{os.path.basename(path)}: {caption}")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Found 25 images


Processing batches:   0%|          | 0/13 [00:00<?, ?it/s]WARNING:root:inference device is not set, using cuda:0, NVIDIA GeForce RTX 3060 Ti
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Processed Batch in 3.06s


...

Processed Batch in 2.00s


Processing batches:  92%|█████████▏| 12/13 [03:06<00:08,  8.10s/it]WARNING:root:inference device is not set, using cuda:0, NVIDIA GeForce RTX 3060 Ti


Processed Batch in 2.43s


Processing batches: 100%|██████████| 13/13 [03:08<00:00, 14.48s/it]

Processed Batch in 2.20s

=== Captions ===
a car_1 - Copy.png: A silver car is driving down a city street with palm trees and buildings in the background.
a car_1.png: A silver car is driving down a street with people and palm trees in the background.
a car_2 - Copy.png: A boy in a red shirt stands in front of a car with a red stripe on the side.
a car_2.png: A boy in a red shirt is walking on the sidewalk next to a car with a red stripe.
a car_3.png: A man in a suit is helping another man out of a car that has a roof rack on top.
a car_4.png: A green car is parked on the side of a street in front of a building.
a car_5.png: A red, white, and blue car is driving down a street.
a car_6.png: A white car is parked in a grassy field.
a car_7.png: A blue car with a yellow bike attached to it is parked on the street.
a car_8.png: A car with a colorful paint job and a sign that reads "A.R.C.T.I.R.E.M.E.N.G.E.T.A.L.A.T.I.N.G.E.T.A.L.A.T.I.N.G.E.T.A.L.A.T.I.N.G.E.T.A.L.A.T.I.N.G.E.T.A.L.A.T.I.N

## Model 3 - llava-hf/llava-onevision-qwen2-0.5b-si-hf

- This is another llava model variant available on [hugging face](https://huggingface.co/llava-hf/llava-onevision-qwen2-0.5b-si-hf) supposedly achieving better overall performance against existing 7B models such as LLaVA-1.5 and Qwen-VL.

- This variant does further support **flash_attention** to enhance execution time however I didn't manage to get this working.

- The [llava-hf/llava-onevision-qwen2-7b-si-hf](https://huggingface.co/llava-hf/llava-onevision-qwen2-7b-si-hf) model was also tested however being a larger scale model performance was worse.

- **NOT VIABLE**: Model takes 3 minutes to process 25 images using an batch_size=2 (larger batch sizes slow down inference due to model scale).

In [ ]:
import torch
from transformers import AutoProcessor, LlavaOnevisionForConditionalGeneration, BitsAndBytesConfig
from PIL import Image
import os
import time
from torch import inference_mode, autocast
import torch

# ------------------------------- Configuration -------------------------------
image_dir = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\images"
device = "cuda" if torch.cuda.is_available() else "cpu"
question = "Describe this image in one sentence."
max_new_tokens = 100
batch_size = 2  # adjust depending on GPU memory

# ------------------------------- Load Model -------------------------------
model_id = "llava-hf/llava-onevision-qwen2-0.5b-si-hf" #"llava-hf/llava-onevision-qwen2-7b-si-hf"
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,  
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"        
)

model = LlavaOnevisionForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    dtype=torch.float16,
    low_cpu_mem_usage=True
    # use_flash_attention_2=True
).to(device)


model = torch.compile(model, mode="reduce-overhead")

processor = AutoProcessor.from_pretrained(model_id, use_fast=True)

# ------------------------------- Load Images -------------------------------
def load_images_from_folder(folder):
    exts = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
    images = []
    paths = []
    for f in os.listdir(folder):
        if f.lower().endswith(exts):
            path = os.path.join(folder, f)
            try:
                img = Image.open(path).convert("RGB")
                images.append(img)
                paths.append(path)
            except Exception as e:
                print(f"Skipping {path}: {e}")
    return images, paths

images, paths = load_images_from_folder(image_dir)
print(f"Loaded {len(images)} images")

# ------------------------------- Generate Captions in Batches -------------------------------
captions = []
model.eval()
with torch.no_grad():
    for i in range(0, len(images), batch_size):
        batch_images = images[i:i+batch_size]
        batch_paths = paths[i:i+batch_size]

        # Prepare conversation for all images in the batch
        conversation = [{
            "role": "user",
            "content": [{"type": "text", "text": question}] + [{"type": "image"} for _ in batch_images]
        }]
        prompt = processor.apply_chat_template(conversation, add_generation_prompt=True)

        start = time.time()
        # Processor handles multiple images at once
        inputs = processor(images=batch_images, text=prompt, return_tensors="pt").to(model.device, torch.float16)

        with inference_mode():
            with autocast(device_type="cuda", dtype=torch.float16):
                outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        
        # Decode each output
        batch_captions = [processor.decode(out[2:], skip_special_tokens=True) for out in outputs]

        captions.extend(zip(batch_paths, batch_captions))
        print(f"Processed batch {i//batch_size+1} in {time.time()-start:.2f}s")

# ------------------------------- Show results -------------------------------
print("\n=== Captions ===")
for path, cap in captions:
    print(f"{os.path.basename(path)}: {cap}")

## Model 4 - Salesforce/blip2-flan-t5-xl

- The [Salesforce/blip2-flan-t5-xl](https://huggingface.co/Salesforce/blip2-flan-t5-xl) model is an **image captioning model** as opposed to the llava model which is a Multimodal Large Language Model. The focus of this model is to caption images hence it can't generate short vs long captions as it doesn't have the capability to follow instructions.

- This model produces captions similar to the short captions seen in the ["Understanding Bias in Large-Scale Visual Datasets"](https://arxiv.org/pdf/2412.01876v1) paper offering a basic description of the image contents

- The model supports only 8-bit qunatization as applying 4bit quantization by uncommenting the code below results in the model producing gibberish in terms of captions.

- **NOT VIABLE**: Uisng 8-bit quantization isn't sufficient to significantly speed up execution as the model still takes 2.28s to process 25 images using a batch_size=25 (larger batch sizes are supported due to model scale being small).


In [17]:
import os
import time
from PIL import Image
import torch
from transformers import Blip2Processor, Blip2ForConditionalGeneration, BitsAndBytesConfig

# ---------------- Settings ----------------
image_dir = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\images"
batch_size = 25
# device is defined but device_map="auto" will handle device assignment
device = "cuda" if torch.cuda.is_available() else "cpu" 
caption_type = "short"  # "short" or "long"

# ---------------- Prompt templates ----------------
short_prompt = "Describe this image in one sentence."
long_prompt = "Describe this image in one paragraph."
custom_prompt = short_prompt if caption_type == "short" else long_prompt

# ---------------- Model setup ----------------
processor = Blip2Processor.from_pretrained("Salesforce/blip2-flan-t5-xl", use_fast = True)

# quantization_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.float16 # <--- Requires float16 inputs for computation
# )

quantization_config = BitsAndBytesConfig(
    load_in_8bit=True,
)

model = Blip2ForConditionalGeneration.from_pretrained(
    "Salesforce/blip2-flan-t5-xl",
    quantization_config = quantization_config,
    use_safetensors=True,
    device_map="auto"
)

# ---------------- Max tokens ----------------
max_length = 100

# ---------------- Collect image paths ----------------
image_paths = [
    os.path.join(image_dir, f)
    for f in os.listdir(image_dir)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
]

print(f"Found {len(image_paths)} images.")

# ---------------- Loop through images in batches ----------------
start_time = time.time()
total_images = 0

for i in range(0, len(image_paths), batch_size):
    batch_paths = image_paths[i:i+batch_size]
    images = []
    valid_paths = []

    for path in batch_paths:
        try:
            img = Image.open(path).convert("RGB")
            images.append(img)
            valid_paths.append(path)
        except Exception as e:
            print(f"Skipping {path}, error: {e}")
            continue

    if not images:
        continue

    # Preprocess batch with instruction prompt (tensors start on CPU)
    inputs = processor(
        images=images,
        text=[custom_prompt] * len(images),
        return_tensors="pt"
    )

    # Move tensors to the appropriate device (GPU) and convert pixel_values to float16
    for k, v in inputs.items():
        if v is not None:
            # Determine the target device (e.g., cuda:0)
            target_device = torch.device(device) 

            # Move tensor to the device
            v = v.to(target_device) 
            
            # If it's a floating-point tensor (the image pixel values), 
            # convert it to float16, as required by bnb_4bit_compute_dtype
            if v.dtype == torch.float32:
                 v = v.to(torch.float16)
            
            inputs[k] = v
            
    # Generate captions
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_length
        )
        captions = [processor.decode(g, skip_special_tokens=True) for g in output_ids]

    # Print results
    for path, caption in zip(valid_paths, captions):
        print(f"{os.path.basename(path)} -> {caption}")

    total_images += len(valid_paths)

end_time = time.time()
elapsed = end_time - start_time
print(f"\nProcessed {total_images} images in {elapsed:.2f} seconds ({total_images/elapsed:.2f} imgs/sec)")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Found 25 images.
a car_1 - Copy.png -> a silver car drives down a street in a city
a car_1.png -> a vintage car drives down a city street
a car_2 - Copy.png -> a car parked on a street
a car_2.png -> a car parked on a street
a car_3.png -> a man is standing next to a car with a hat on
a car_4.png -> a green car parked on a street
a car_5.png -> a red and white car driving down a road
a car_6.png -> a white car parked in a field
a car_7.png -> a blue car with a flaming flame on the side
a car_8.png -> a car with a tiger on the bonnet
a car_9.png -> a man sits on a car seat in front of a car
a girl_1.png -> a girl in a twirls around a frame in a framed frame
a girl_3.png -> a colorful office with a mural on the wall
a girl_4.png -> a pair of people in a room with a mirror
a girl_5.png -> a collage of images of a street scene in a city
a girl_7.png -> a painting of a room with a window
a girl_8.png -> a drawing of a building with a doorway and windows
a_car_0.png -> a car driving down a r

## Model 5 - vikhyatk/moondream2

- **NOT VIABLE**: Model doesn't appear to support batching

In [4]:
# Moondream 2 doesn't seem to support batching

from transformers import AutoModelForCausalLM
from PIL import Image
import torch

# Load the model
model = AutoModelForCausalLM.from_pretrained(
    "vikhyatk/moondream2",
    trust_remote_code=True,
    dtype=torch.bfloat16,
    device_map="cuda",
)

# Load your image
image = Image.open(r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\images\a girl_4.png")

# Optionally set sampling settings
# settings = {"temperature": 0.5, "max_tokens": 768, "top_p": 0.3}
settings = {"temperature": 0.5, "max_tokens": 100, "top_p": 0.3}

# Generate a short caption
short_result = model.caption(
    image, 
    length="short", 
    settings=settings
)
print(short_result)

{'caption': 'Two black and white photographs of two silhouetted individuals are displayed on a beige wall, with the left image slightly higher than the right.'}


## Model 6 - Salesforce/blip-image-captioning-large

- The [Salesforce/blip-image-captioning-large](https://huggingface.co/Salesforce/blip-image-captioning-large) is a predecessor to the [Salesforce/blip2-flan-t5-xl](https://huggingface.co/Salesforce/blip2-flan-t5-xl) model being an **image captioning model** as opposed to the llava model which is a Multimodal Large Language Model. The focus of this model is to caption images hence it can't generate short vs long captions as it doesn't have the capability to follow instructions.

- This model produces more basic captions than its predecessor however the captions are similar to the short captions seen in the ["Understanding Bias in Large-Scale Visual Datasets"](https://arxiv.org/pdf/2412.01876v1) paper offering a basic description of the image contents.

- The current implemented approch does not use quantization as inference speed is sufficient as is.

- **PAPER**: https://arxiv.org/abs/2201.12086

- **VIABLE**: Model takes 2.28s to process 25 images using a batch_size=25 (larger batch sizes are supported due to model scale being small).

In [1]:
import os
import time
from PIL import Image
import torch
from transformers import BlipProcessor, BlipForConditionalGeneration

# ---------------- Settings ----------------
image_dir = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\images"
batch_size = 50  # Adjust based on GPU memory
device = "cuda" if torch.cuda.is_available() else "cpu"

# ---------------- Prefix template ----------------
output_prefix = "An image of"

# ---------------- Model setup ----------------
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-large", use_fast=True)
model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-large",
    use_safetensors=True
)
model = model.to(device)
model.eval()

# ---------------- Max tokens ----------------
max_length = 100

# ---------------- Collect image paths ----------------
image_paths = [os.path.join(image_dir, f) for f in os.listdir(image_dir)
               if f.lower().endswith((".jpg", ".jpeg", ".png"))] * 4

print(f"Found {len(image_paths)} images.")

# ---------------- Loop through images in batches ----------------
start_time = time.time()
total_images = 0

for i in range(0, len(image_paths), batch_size):
    batch_paths = image_paths[i:i+batch_size]
    images = []
    valid_paths = []

    for path in batch_paths:
        try:
            img = Image.open(path).convert("RGB")
            images.append(img)
            valid_paths.append(path)
        except Exception as e:
            print(f"Skipping {path}, error: {e}")
            continue

    if not images:
        continue

    # Preprocess batch WITHOUT a literal prompt
    inputs = processor(
        images=images, 
        text=[output_prefix]*len(images),  # pass the instruction
        return_tensors="pt"
    ).to(device, torch.float16)

    # Generate captions (length controlled by max_length)
    with torch.no_grad():
        out = model.generate(**inputs, 
                             max_new_tokens=max_length, 
                             num_beams=1, 
                             do_sample=True,          # Optional: Increase inference time for more creative outputs
                             top_k=50,                # Optional: Control word choices
                             temperature=0.7          # Optional: Control randomness (0.7-0.9 is good)
                            )
        captions = processor.batch_decode(out, skip_special_tokens=True)

    # Print results
    for path, caption in zip(valid_paths, captions):
        print(f"{os.path.basename(path)} -> {caption}")

    total_images += len(valid_paths)

elapsed = time.time() - start_time
print(f"\nProcessed {total_images} images in {elapsed:.2f} seconds ({total_images/elapsed:.2f} imgs/sec)")

Skipping import of cpp extensions due to incompatible torch version 2.6.0+cu124 for torchao version 0.14.0         Please see GitHub issue #2919 for more info


Found 100 images.
a car_1 - Copy.png -> an image of a silver van is parked next to a tall building
a car_1.png -> an image of a car driving down the road with many people
a car_2 - Copy.png -> an image of a car that is parked in front of a building
a car_2.png -> an image of a silver car traveling down a street next to a tree
a car_3.png -> an image of two people fixing a car on the side of the road
a car_4.png -> an image of a green car is parked on the side of the road
a car_5.png -> an image of a car painted in red, white and blue paint
a car_6.png -> an image of a white, classic american car sitting in a field
a car_7.png -> an image of a blue car that is painting on the side of a road
a car_8.png -> an image of a yellow car with a surfboard on the front of it
a car_9.png -> an image of a man leaning on a car parked on a street
a girl_1.png -> an image of a girl is standing in a mirror with her arms out
a girl_3.png -> an image of a long hallway in a building with a couple of chair

## Model 7 - microsoft/Florence-2-base-ft

- The [microsoft/Florence-2-base-ft](https://huggingface.co/microsoft/Florence-2-base-ft) is a vision foundation model that uses a prompt-based approach to handle a wide range of vision and vision-language tasks. Florence-2 can interpret simple text prompts to perform tasks like captioning, object detection, and segmentation.

The larger [microsoft/Florence-2-base-ft](https://huggingface.co/microsoft/Florence-2-base-ft) was also tested with 4-bit quantization howeve it takes 7 sec for 25 images

- This model supports the concept of **\<CAPTION\>** & **\<DETAILED_CAPTION\>** which outline the lenght and detail the model should provide in its captions similar to the the short vs long captions in the  ["Understanding Bias in Large-Scale Visual Datasets"](https://arxiv.org/pdf/2412.01876v1) paper.

- Their exists two size varients the base & large however the latter is much more intensive and takes to long to carry out inference on 25 images.

- The current implemented approch does not use quantization as inference speed is sufficient as is.

- **PAPER**: https://arxiv.org/abs/2311.06242

- **VIABLE**: Model takes 2.55 to process 25 images using a batch_size=25 (larger batch sizes are supported due to model scale being small).

In [ ]:
# VIABLE MODEL PROCESSES 200 IMAGES IN 24 SEC
import os
os.environ["ATTN_IMPLEMENTATION"] = "eager"
os.environ["TRANSFORMERS_NO_SDPA"] = "1"

import torch
from transformers import AutoProcessor, AutoModelForCausalLM, BitsAndBytesConfig
from PIL import Image
import numpy as np
from tqdm import tqdm
import time

# -------------------------------
# 1. Configuration
# -------------------------------
image_dir = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\images"
batch_size = 25                # Adjust for GPU VRAM (Florence-2 is heavy)
max_new_tokens = 100           # Length of captions
prompt_text = "<DETAILED_CAPTION>" #"<CAPTION>" 

device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

np.float_ = np.float64
np.complex_ = np.complex128

# # Quantization config (4-bit)
# quant_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_compute_dtype=torch.float16,
#     bnb_4bit_use_double_quant=True,
#     bnb_4bit_quant_type="nf4"
# )

# -------------------------------
# 2. Load model and processor
# -------------------------------
model_name = "microsoft/Florence-2-base-ft" #"microsoft/Florence-2-large-ft"
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch_dtype,
    trust_remote_code=True,
    attn_implementation="eager",
    # quantization_config=quant_config
).to(device).eval()

processor = AutoProcessor.from_pretrained(model_name, trust_remote_code=True)

# -------------------------------
# 3. Helper to load images
# -------------------------------
def load_images_from_dir(directory):
    exts = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
    return [os.path.join(directory, f) for f in os.listdir(directory) if f.lower().endswith(exts)]

image_paths = load_images_from_dir(image_dir)* 8
print(f"Found {len(image_paths)} images")

# -------------------------------
# 4. Batched inference
# -------------------------------
all_captions = []

for i in tqdm(range(0, len(image_paths), batch_size), desc="Processing batches"):
    batch_paths = image_paths[i:i + batch_size]
    images = [Image.open(p).convert("RGB") for p in batch_paths]

    inputs = processor(
        text=[prompt_text] * len(images),
        images=images,
        return_tensors="pt",
        padding=True
    ).to(device, torch_dtype)

    start_time = time.time()
    with torch.no_grad():
        generated_ids = model.generate(
            input_ids=inputs["input_ids"],
            pixel_values=inputs["pixel_values"],
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=1,       # Beam search off
            use_cache=False    # Disable caching - otherwise crashes the model
        )

    decoded = processor.batch_decode(generated_ids, skip_special_tokens=True)

    for path, text, img in zip(batch_paths, decoded, images):
        caption = processor.post_process_generation(
            text,
            task=prompt_text,
            image_size=(img.width, img.height)
        )
        all_captions.append((os.path.basename(path), caption))

    print(f"Batch {i//batch_size + 1}: {len(batch_paths)} images processed in {time.time() - start_time:.2f}s")
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

# -------------------------------
# 5. Output results
# -------------------------------
print("\n=== Captions ===")
for img_name, caption in all_captions:
    print(f"{img_name}: {caption[prompt_text]}")


Skipping import of cpp extensions due to incompatible torch version 2.6.0+cu124 for torchao version 0.14.0         Please see GitHub issue #2919 for more info


Found 3000 images


Processing batches:   1%|          | 1/120 [00:03<07:00,  3.53s/it]

Batch 1: 25 images processed in 2.74s


Processing batches:   2%|▏         | 2/120 [00:06<06:14,  3.17s/it]

...

Batch 37: 25 images processed in 2.32s


Processing batches:  32%|███▏      | 38/120 [01:50<04:04,  2.98s/it]

Batch 38: 25 images processed in 2.00s


Processing batches:  32%|███▎      | 39/120 [01:52<03:55,  2.91s/it]

Batch 39: 25 images processed in 1.95s


## Model 8 - microsoft/kosmos-2-patch14-224

- The [microsoft/kosmos-2-patch14-224](https://huggingface.co/microsoft/kosmos-2-patch14-224) model is a Grounding Multimodal Large Language Models.

- This model is capable of performing [different tasks](https://huggingface.co/microsoft/kosmos-2-patch14-224#:~:text=Here%20are%20the%20tasks%20Kosmos%2D2%20could%20perform) through changing the prompts:
    - Phrase Grounding
    - Referring Expression Comprehension
    - Referring expression generation
    - Grounded VQA
    - Grounded VQA with multimodal referring via bounding boxes
    - Brief
    - Detailed

- Model supports short vs long prompts via the prompts **"\<grounding\> An image of"** and **"\<grounding\> Describe this image in detail:"** respectively.

- **PAPER**: https://arxiv.org/abs/2306.14824

- **VIABLE**: Model takes 15s to process 200 images using a batch_size=200.

In [ ]:
# VIABLE MODEL PROCESSES 200 IMAGES IN 15 SEC
import torch
from transformers import AutoProcessor, Kosmos2ForConditionalGeneration, BitsAndBytesConfig
from PIL import Image
import os, time
from tqdm import tqdm

# -------------------------------
# 1. Configuration
# -------------------------------
image_dir = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\images"
batch_size = 25                # Adjust for GPU VRAM (try 2–8 for 8GB GPU)
max_new_tokens = 100                # Lower = faster; higher = more descriptive captions

# Quantization config (4-bit)
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

# -------------------------------
# 2. Load model and processor
# -------------------------------
model_name = "microsoft/kosmos-2-patch14-224"
device = "cuda" if torch.cuda.is_available() else "cpu"

model = Kosmos2ForConditionalGeneration.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=quant_config
).eval()

processor = AutoProcessor.from_pretrained(model_name, use_fast=True)

# -------------------------------
# 3. Helper to load images
# -------------------------------
def load_images_from_dir(directory):
    exts = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
    image_files = [os.path.join(directory, f) for f in os.listdir(directory)
                   if f.lower().endswith(exts)]
    return image_files

image_paths = load_images_from_dir(image_dir)[:3000] #* 40
print(f"Found {len(image_paths)} images")

# -------------------------------
# 4. Batched inference
# -------------------------------
all_captions = []

for i in tqdm(range(0, len(image_paths), batch_size), desc="Processing batches"):
    batch_paths = image_paths[i:i + batch_size]
    images = [Image.open(p).convert("RGB") for p in batch_paths]

    prompt = "<grounding> Describe this image in detail:" # "<grounding> An image of"
    inputs = processor(text=[prompt] * len(images), images=images, return_tensors="pt", padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    start_time = time.time()
    with torch.no_grad():
        outputs = model.generate(
            pixel_values=inputs["pixel_values"],
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            image_embeds_position_mask=inputs["image_embeds_position_mask"],
            max_new_tokens=max_new_tokens,
            use_cache=True
        )

    decoded = processor.batch_decode(outputs, skip_special_tokens=True)
    for path, text in zip(batch_paths, decoded):
        caption, _ = processor.post_process_generation(text)
        all_captions.append((os.path.basename(path), caption))

    print(f"Batch {i//batch_size + 1}: {len(batch_paths)} images processed in {time.time() - start_time:.2f}s")
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

# -------------------------------
# 5. Output results
# -------------------------------
print("\n=== Captions ===")
for img_name, caption in all_captions:
    print(f"{img_name}: {caption}")


Skipping import of cpp extensions due to incompatible torch version 2.6.0+cu124 for torchao version 0.14.0         Please see GitHub issue #2919 for more info


Found 3000 images


Processing batches:   1%|          | 1/120 [00:07<15:23,  7.76s/it]

Batch 1: 25 images processed in 7.58s


Processing batches:   2%|▏         | 2/120 [00:14<14:15,  7.25s/it]

...

Batch 28: 25 images processed in 6.21s


Processing batches:  24%|██▍       | 29/120 [03:10<09:44,  6.42s/it]

Batch 29: 25 images processed in 6.20s


Processing batches:  25%|██▌       | 30/120 [03:18<10:03,  6.71s/it]

Batch 30: 25 images processed in 7.26s


## Model 9 - nlpconnect/vit-gpt2-image-captioning

- The [nlpconnect/vit-gpt2-image-captioning](https://huggingface.co/nlpconnect/vit-gpt2-image-captioning) model is an image captioning model.

- **PAPER**: https://ankur3107.github.io/blogs/the-illustrated-image-captioning-using-transformers/

- **VIABLE**: Model takes 4s to process 200 images using a batch_size=25, however the captions are very basic and not as detaild in comparison to other small scale models.

In [1]:
import torch
from transformers import VisionEncoderDecoderModel, ViTImageProcessor, AutoTokenizer
from PIL import Image
import os, time
from tqdm import tqdm

# -------------------------------
# 1. Configuration
# -------------------------------
image_dir = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\images"
batch_size = 25                 # Adjust to GPU VRAM (8GB → small batches)
max_length = 100                 # Caption length

device = "cuda" if torch.cuda.is_available() else "cpu"

# -------------------------------
# 2. Load model and processor
# -------------------------------
model_name = "nlpconnect/vit-gpt2-image-captioning"
model = VisionEncoderDecoderModel.from_pretrained(model_name).to(device).eval()
processor = ViTImageProcessor.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# -------------------------------
# 3. Helper to load images
# -------------------------------
def load_images_from_dir(directory):
    exts = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
    return [os.path.join(directory, f) for f in os.listdir(directory) if f.lower().endswith(exts)]

image_paths = load_images_from_dir(image_dir) * 8
print(f"Found {len(image_paths)} images")

# -------------------------------
# 4. Batched inference
# -------------------------------
all_captions = []

for i in tqdm(range(0, len(image_paths), batch_size), desc="Processing batches"):
    batch_paths = image_paths[i:i + batch_size]
    images = [Image.open(p).convert("RGB") for p in batch_paths]

    # Preprocess images
    pixel_values = processor(images=images, return_tensors="pt").pixel_values.to(device)

    start_time = time.time()
    with torch.no_grad():
        output_ids = model.generate(pixel_values, max_length=max_length, num_beams=3)

    captions = tokenizer.batch_decode(output_ids, skip_special_tokens=True)

    for path, caption in zip(batch_paths, captions):
        all_captions.append((os.path.basename(path), caption))

    print(f"Batch {i//batch_size + 1}: {len(batch_paths)} images processed in {time.time() - start_time:.2f}s")
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

# -------------------------------
# 5. Output results
# -------------------------------
print("\n=== Captions ===")
for img_name, caption in all_captions:
    print(f"{img_name}: {caption}")


Skipping import of cpp extensions due to incompatible torch version 2.6.0+cu124 for torchao version 0.14.0         Please see GitHub issue #2919 for more info


Found 200 images


Processing batches:   0%|          | 0/8 [00:00<?, ?it/s]The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
We strongly recommend passing in an `attention_mask` since your input_ids may be padded. See https://huggingface.co/docs/transformers/troubleshooting#incorrect-output-when-padding-tokens-arent-masked.
You may ignore this warning if your `pad_token_id` (50256) is identical to the `bos_token_id` (50256), `eos_token_id` (50256), or the `sep_token_id` (None), and your input is not padded.
Processing batches:  12%|█▎        | 1/8 [00:01<00:10,  1.43s/it]

Batch 1: 25 images processed in 0.97s


Processing batches:  25%|██▌       | 2/8 [00:02<00:07,  1.24s/it]

...

Batch 6: 25 images processed in 0.65s


Processing batches:  88%|████████▊ | 7/8 [00:07<00:01,  1.10s/it]

Batch 7: 25 images processed in 0.65s


Processing batches: 100%|██████████| 8/8 [00:09<00:00,  1.13s/it]

Batch 8: 25 images processed in 0.65s

=== Captions ===
a car_1 - Copy.png: a street view of a train going down the tracks 
a car_1.png: a car driving down a street next to tall buildings 
a car_2 - Copy.png: a car is parked on the side of the road 
a car_2.png: a car is parked on the side of the road 
a car_3.png: a man and a woman are standing in front of a truck 
a car_4.png: a green car is parked in front of a building 
a car_5.png: a red and white car parked in front of a building 
a car_6.png: a white car parked in a grassy field 
a car_7.png: a car is parked next to a bicycle on the street 
a car_8.png: a vintage car with a picture of a man on it 
a car_9.png: a white car is parked in front of a building 
a girl_1.png: a woman standing in front of a mirror in a room 
a girl_3.png: a painting of a mannequin on a wall 
a girl_4.png: two pictures of people standing in front of a wall 
a girl_5.png: a collage of photos of a living room with furniture 
a girl_7.png: a painting of a w

## Model 10 - cnmoro/tiny-image-captioning

- The [cnmoro/tiny-image-captioning](https://huggingface.co/cnmoro/tiny-image-captioning) model is an image captioning model, based on bert-tiny and vit-small, weighing only 100mb. 

- This model is extremly small scale and only being considered for the worst case scenario where no other models are viable.

- **PAPER**: No Paper Available

- **VIABLE**: Model takes 1s to process 200 images using a batch_size=200, however the captions are very basic and appear to be not as accurate to model content as other valid models.

In [2]:
import torch
from transformers import VisionEncoderDecoderModel, AutoTokenizer, AutoImageProcessor
from PIL import Image
import os, time
from tqdm import tqdm

# -------------------------------
# 1. Configuration
# -------------------------------
image_dir = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\images"
batch_size = 200
device = "cuda" if torch.cuda.is_available() else "cpu"
max_length = 64

# -------------------------------
# 2. Load model, tokenizer, and processor
# -------------------------------
model_name = "cnmoro/tiny-image-captioning"
model = VisionEncoderDecoderModel.from_pretrained(model_name).to(device).eval()
tokenizer = AutoTokenizer.from_pretrained(model_name)
image_processor = AutoImageProcessor.from_pretrained(model_name)

# -------------------------------
# 3. Helper to load images
# -------------------------------
def load_images_from_dir(directory):
    exts = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
    return [os.path.join(directory, f) for f in os.listdir(directory) if f.lower().endswith(exts)]

image_paths = load_images_from_dir(image_dir) * 8
print(f"Found {len(image_paths)} images")

# -------------------------------
# 4. Batched inference
# -------------------------------
all_captions = []

for i in tqdm(range(0, len(image_paths), batch_size), desc="Processing batches"):
    batch_paths = image_paths[i:i + batch_size]
    images = [Image.open(p).convert("RGB") for p in batch_paths]

    # Preprocess images
    pixel_values = image_processor(images, return_tensors="pt").pixel_values.to(device)

    start_time = time.time()
    with torch.no_grad():
        generated_ids = model.generate(
            pixel_values,
            max_length=max_length,
            num_beams=3,      # 1 for faster but slightly lower quality
            temperature=0.7,
            top_p=0.8,
            top_k=50
        )

    captions = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)

    for path, caption in zip(batch_paths, captions):
        all_captions.append((os.path.basename(path), caption))

    print(f"Batch {i//batch_size + 1}: {len(batch_paths)} images processed in {time.time() - start_time:.2f}s")
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

# -------------------------------
# 5. Output results
# -------------------------------
print("\n=== Captions ===")
for img_name, caption in all_captions:
    print(f"{img_name}: {caption}")


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Found 200 images


Processing batches:   0%|          | 0/1 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Both `max_new_tokens` (=25) and `max_length`(=64) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
We strongly recommend passing in an `attention_mask` since your input_ids may be padded. See https://huggingface.co/docs/transformers/troubleshooting#incorrect-output-when-padding-tokens-arent-masked.
Processing batches: 100%|██████████| 1/1 [00:04<00:00,  4.50s/it]

Batch 1: 200 images processed in 0.81s

=== Captions ===
a car_1 - Copy.png: a man is sitting on the sidewalk.
a car_1.png: a man is walking through the street.
a car_2 - Copy.png: a man is walking through the street.
a car_2.png: a man is walking through the street.
a car_3.png: a man is sitting on the sidewalk.
a car_4.png: a man is walking through the street.
a car_5.png: a yellow car is running through the water.
a car_6.png: a man is walking through a wooded area.
a car_7.png: a man is sitting on the street.
a car_8.png: a yellow car is stopped in front of a car.
a car_9.png: a man is sitting on the street.
a girl_1.png: a woman is sitting in front of a building.
a girl_3.png: a man wearing a blue shirt is sitting on the floor.
a girl_4.png: a group of people are in front of a building.
a girl_5.png: a man is sitting on a bench.
a girl_7.png: a man is sitting in front of a building.
a girl_8.png: a man is sitting on a bench.
a_car_0.png: a man is walking through a street.
a_girl_0

# Testing VIABLE Models 

- This test will determine the best model in terms of speed, output quality and additional functionality (were applicable).

- Testing will be carried out on the **Flickr30K** & **COCO** Datasets using the procedure outlined in [Improving Image Captioning Descriptiveness by Ranking and LLM-based Fusion](https://arxiv.org/html/2306.11593) via the **XXX** metrics (This might change as the metrics don't reflect what we want)

### Relevant Papers:

- [Improving Image Captioning Descriptiveness by Ranking and LLM-based Fusion](https://arxiv.org/html/2306.11593)

## Downloading the COCO2014 & Flickr30K datasets

- These datasets were filtered to utilse the Karpathy Splits, which are commonly used splits for image captioning test as per the [Improving Image Captioning Descriptiveness by Ranking and LLM-based Fusion](https://arxiv.org/pdf/2306.11593v3) paper.

    - COCO Images - 2014 Val Images downloaded from [link](https://cocodataset.org/#download).
    - Flickr30l Images - Downloade from [link](https://huggingface.co/datasets/nlphuji/flickr30k/blob/main/flickr30k-images.zip).
        - The Karpathy Splits were downloaded from [link](https://github.com/Delphboy/karpathy-splits/tree/main).

In [5]:
import os
import json
import shutil
from tqdm import tqdm

def copy_images_from_json(json_path, src_dir, dst_dir, split_filter=None):
    """
    Copies images from JSON. Optionally filter by split ('train', 'val', 'test').
    """
    import os, json, shutil
    from tqdm import tqdm

    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    images = data.get("images", [])
    if split_filter:
        images = [img for img in images if img.get("split") == split_filter]

    os.makedirs(dst_dir, exist_ok=True)

    copied, missing = 0, 0
    for img_info in tqdm(images, desc=f"Copying {split_filter or 'all'} images"):
        filename = img_info.get("filename")
        filepath = img_info.get("filepath", "")
        src_path = os.path.join(src_dir, filepath, filename) if filepath else os.path.join(src_dir, filename)
        dst_path = os.path.join(dst_dir, filename)

        if os.path.exists(src_path):
            shutil.copy2(src_path, dst_path)
            copied += 1
        else:
            missing += 1
            print(f"Missing: {src_path}")

    print(f"\nCopied {copied} {split_filter or 'all'} images to {dst_dir}")
    if missing:
        print(f"{missing} missing images")

In [4]:
# ---------------- COCO Karpathy Split ----------------
coco_json = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\COCO2014\dataset_coco.json"
coco_src = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\COCO2014"
coco_dst = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\COCO2014\val2014_karpathy_split"
copy_images_from_json(coco_json, coco_src, coco_dst, split_filter="test")

Copying test images: 100%|██████████| 5000/5000 [00:54<00:00, 91.54it/s] 



✅ Copied 5000 test images to C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\COCO2014\val2014_karpathy_split


In [ ]:
# ---------------- Flickr30k Karpathy Split ----------------
flickr_json = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\Flickr30K\dataset_flickr32k.json"
flickr_src = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\Flickr30K\flickr30k-images"
flickr_dst = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\Flickr30K\flickr30k-images_karpathy_split"
copy_images_from_json(flickr_json, flickr_src, flickr_dst, split_filter="test")

Copying test images: 100%|██████████| 1000/1000 [00:01<00:00, 500.48it/s]


Copied 1000 test images to C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\flickr30k-images_karpathy_split


In [9]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T
import numpy as np
import torch
import os

# ---------------- Dataset ----------------
class ImageDataset(Dataset):
    def __init__(self, image_dir: str, extensions=(".jpg", ".jpeg", ".png"), as_tensor=False, transform=None):
        self.image_paths = [
            os.path.join(image_dir, f)
            for f in os.listdir(image_dir)
            if f.lower().endswith(extensions)
        ]
        self.as_tensor = as_tensor
        self.transform = transform or (T.ToTensor() if as_tensor else None)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = self.image_paths[idx]
        img = Image.open(path).convert("RGB")

        if self.as_tensor and self.transform:
            img = self.transform(img)

        return img, path

# ---------------- Collate Function ----------------
def collate_images(batch):
    imgs, paths = zip(*batch)
    return list(imgs), list(paths)

# ---------------- Prefetching Loader ----------------
class PrefetchLoader:
    """Wrap a DataLoader to prefetch the next batch to GPU asynchronously."""
    def __init__(self, loader, device, as_tensor=False):
        self.loader = iter(loader)
        self.device = device
        self.as_tensor = as_tensor
        self.stream = torch.cuda.Stream() if device == "cuda" else None
        self.next_batch = None
        self._prefetch()

    def _prefetch(self):
        try:
            imgs, paths = next(self.loader)
        except StopIteration:
            self.next_batch = None
            return

        # Prefetching logic
        if self.device == "cuda":
            with torch.cuda.stream(self.stream):
                if self.as_tensor:
                    # Move tensors to GPU asynchronously
                    imgs = [img.to(self.device, non_blocking=True) for img in imgs]
                # else: keep PIL images on CPU (LLaVA/CLIP-style models expect CPU PILs)
        else:
            if self.as_tensor:
                imgs = [img.to(self.device) for img in imgs]

        self.next_batch = (imgs, paths)

    def __iter__(self):
        return self

    def __next__(self):
        if self.next_batch is None:
            raise StopIteration
        batch = self.next_batch
        self._prefetch()
        return batch

# ---------------- DataLoader setup ----------------
def get_image_loader(image_dir: str, batch_size=4, num_workers=4, device='cpu', as_tensor=False):
    dataset = ImageDataset(image_dir, as_tensor=as_tensor)
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        num_workers=num_workers,
        shuffle=False,
        collate_fn=collate_images,
        pin_memory=True
    )
    prefetch_loader = PrefetchLoader(loader, device=device, as_tensor=as_tensor)
    return prefetch_loader, len(dataset)

## Salesforce/blip-image-captioning-large

In [ ]:
"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\.venv - Copy - Copy\Scripts\python.exe" "C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioning\blip-image-captioning-large.py" --image_dir "C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\Flickr30k\flickr30k-images_karpathy_split" --batch_size 8 --num_workers 8 --output_file "C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\Flickr30k\blip_image_captioning_large_flickr30k_images_karpathy_split_captions.json" --prompt "An image of"
# 2 Mins 30 Sec

"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\.venv-Copy-Copy\Scripts\python.exe" "C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioning\blip-image-captioning-large.py" --image_dir "C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\COCO2014\val2014_karpathy_split" --batch_size 8 --num_workers 8 --output_file "C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\COCO2014\blip_image_captioning_large_coco2014_karpathy_split_captions.json" --prompt "An image of"
#  9 Mins 4 Sec

## microsoft/Florence-2-base-ft

In [ ]:
"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\.venv - Copy - Copy\Scripts\python.exe" "C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioning\florence-2-base-ft.py" --image_dir "C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\Flickr30k\flickr30k-images_karpathy_split" --batch_size 8 --num_workers 8 --output_file "C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\Flickr30k\florence_2_base_ft_flickr30k_images_karpathy_split_long_captions.json" --prompt "<DETAILED_CAPTION>"
# 4 Mins 4 Sec

# "C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\.venv - Copy - Copy\Scripts\python.exe" "C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioning\florence-2-base-ft.py" --image_dir "C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\Flickr30k\flickr30k-images_karpathy_split" --batch_size 8 --num_workers 8 --output_file "C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\Flickr30k\florence_2_base_ft_flickr30k_images_karpathy_split_short_captions.json" --prompt "<CAPTION>"




"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\.venv-Copy-Copy\Scripts\python.exe" "C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioning\florence-2-base-ft.py" --image_dir "C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\COCO2014\val2014_karpathy_split" --batch_size 8 --num_workers 8 --output_file "C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\COCO2014\florence_2_base_ft_coco2014_karpathy_split_long_captions.json" --prompt "<DETAILED_CAPTION>"
# 14 Mins 7 Sec

# "C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\.venv - Copy - Copy\Scripts\python.exe" "C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioning\florence-2-base-ft.py" --image_dir "C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\COCO2014\val2014images_karpathy_split" --batch_size 8 --num_workers 8 --output_file "C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\COCO2014\florence_2_base_ft_coco2014_karpathy_split_short_captions.json" --prompt "<CAPTION>"

## microsoft/kosmos-2-patch14-224

In [ ]:
"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\.venv - Copy - Copy\Scripts\python.exe" "C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioning\kosmos-2-patch14-224.py" --image_dir "C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\Flickr30k\flickr30k-images_karpathy_split" --batch_size 8 --num_workers 8 --output_file "C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\Flickr30k\kosmos_2_patch14_224_flickr30k_images_karpathy_split_long_captions.json" --prompt "<grounding> Describe this image in detail:"
# 12 Mins 23 Sec

# "C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\.venv - Copy - Copy\Scripts\python.exe" "C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioning\kosmos-2-patch14-224.py" --image_dir "C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\Flickr30k\flickr30k-images_karpathy_split" --batch_size 8 --num_workers 8 --output_file "C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\Flickr30k\kosmos_2_patch14_224_flickr30k_images_karpathy_split_short_captions.json" --prompt "<grounding> An image of"
# X Mins X Sec



"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\.venv-Copy-Copy\Scripts\python.exe" "C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioning\kosmos-2-patch14-224.py" --image_dir "C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\COCO2014\val2014_karpathy_split" --batch_size 8 --num_workers 8 --output_file "C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\COCO2014\kosmos_2_patch14_224_coco2014_karpathy_split_long_captions.json" --prompt "<grounding> Describe this image in detail:"
# 55 Mins 38 Sec

# "C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\.venv - Copy - Copy\Scripts\python.exe" "C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioning\kosmos-2-patch14-224.py" --image_dir "C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\COCO2014\val2014_karpathy_split" --batch_size 8 --num_workers 8 --output_file "C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\COCO2014\kosmos_2_patch14_224_coco2014_karpathy_split_short_captions.json" --prompt "<grounding> An image of"
# X Mins X Sec

## nlpconnect/vit-gpt2-image-captioning

In [ ]:
"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\.venv - Copy - Copy\Scripts\python.exe" "C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioning\vit-gpt2-image-captioning.py" --image_dir "C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\Flickr30k\flickr30k-images_karpathy_split" --batch_size 8 --num_workers 8 --output_file "C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\Flickr30k\vit_gpt2_image_captioning_flickr30k_images_karpathy_split_captions.json"
# 1 Mins 52 sec

"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\.venv - Copy - Copy\Scripts\python.exe" "C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioning\vit-gpt2-image-captioning.py" --image_dir "C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\COCO2014\val2014_karpathy_split" --batch_size 8 --num_workers 8 --output_file "C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\COCO2014\vit_gpt2_image_captioning_coco2014_karpathy_split_captions.json"
# 5 Mins 50 sec

## cnmoro/tiny-image-captioning

In [ ]:
"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\.venv - Copy - Copy\Scripts\python.exe" "C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioning\tiny-image-captioning.py" --image_dir "C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\Flickr30k\flickr30k-images_karpathy_split" --batch_size 8 --num_workers 8 --output_file "C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\Flickr30k\tiny_image_captioning_flickr30k_images_karpathy_split_captions.json"
# 2 Mins 28 sec

"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\.venv - Copy - Copy\Scripts\python.exe" "C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioning\tiny-image-captioning.py" --image_dir "C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\COCO2014\val2014_karpathy_split" --batch_size 8 --num_workers 8 --output_file "C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\COCO2014\tiny_image_captioning_coco2014_karpathy_split_captions.json"
# 8 Mins 12 sec

In [5]:
# !pip install pycocoevalcap
# !pip install nltk

In [20]:
# import nltk
# nltk.download('wordnet')
# nltk.download('omw-1.4')

In [1]:
import json
import numpy as np
from nltk.translate.bleu_score import corpus_bleu, sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from nltk.util import ngrams
from sentence_transformers import SentenceTransformer
from pycocoevalcap.cider.cider import Cider
# from pycocoevalcap.spice.spice import Spice # Uncomment when SPICE is working

try:
    SBERT_MODEL = SentenceTransformer('all-MiniLM-L6-v2', device='cuda')
except Exception:
    # Fallback to CPU if CUDA is unavailable or incompatible
    SBERT_MODEL = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')

def evaluate_captioning_model(coco_ref_file: str, model_caps_file: str, n_gram: int = 4) -> dict:
    """
    Computes various image captioning metrics (BLEU, METEOR, CIDEr, N-gram Diversity, 
    and SBERT Semantic Alignment) between a set of generated captions and ground truth references.

    Args:
        coco_ref_file (str): Path to the ground truth COCO reference JSON file.
        model_caps_file (str): Path to the model's generated captions JSON file.
        n_gram (int): The N-gram value used for BLEU and N-gram Diversity (default is 4).

    Returns:
        dict: A dictionary containing all computed metrics.
    """
    
    print(f"--- Starting Evaluation ---")
    print(f"Ref File: {coco_ref_file}")
    print(f"Caps File: {model_caps_file}\n")
    
    # -----------------------------
    # Load JSON files
    # -----------------------------
    try:
        with open(coco_ref_file, 'r', encoding='utf-8') as f:
            ref_data = json.load(f)

        with open(model_caps_file, 'r', encoding='utf-8') as f:
            model_data = json.load(f)
    except FileNotFoundError as e:
        print(f"Error: File not found: {e}")
        return {}
    except json.JSONDecodeError as e:
        print(f"Error: Invalid JSON format in file: {e}")
        return {}


    # -----------------------------
    # Prepare mappings: imgid -> captions
    # -----------------------------
    # Ground Truths (refs) are a list of 5 raw sentences per image ID
    refs = {img['imgid']: [s['raw'] for s in img['sentences']] for img in ref_data.get('images', [])}
    # Predictions (preds) are usually a single raw sentence per image ID
    preds = {img['imgid']: img['sentences'][0]['raw'] for img in model_data.get('images', [])}
    
    # Use only image IDs present in both files
    img_ids = sorted(list(set(refs.keys()) & set(preds.keys())))
    
    if not img_ids:
        print("Error: No matching image IDs found between the reference and prediction files.")
        return {}

    captions_list = [preds[i] for i in img_ids]
    results = {}

    # ---------------------------------------------
    # 1. Classical metrics: BLEU, METEOR
    # ---------------------------------------------
    print("1. Computing Classical Metrics (BLEU, METEOR)...")
    all_refs = [[r.split() for r in refs[i]] for i in img_ids]
    all_preds = [preds[i].split() for i in img_ids]

    # Initialize smoothing function
    chencherry = SmoothingFunction()

    results['BLEU@1'] = corpus_bleu(all_refs, all_preds, weights=(1,0,0,0), smoothing_function=chencherry.method7)
    results['BLEU@2'] = corpus_bleu(all_refs, all_preds, weights=(0.5,0.5,0,0), smoothing_function=chencherry.method7)
    results['BLEU@3'] = corpus_bleu(all_refs, all_preds, weights=(0.333,0.333,0.333,0), smoothing_function=chencherry.method7)
    results['BLEU@4'] = corpus_bleu(all_refs, all_preds, weights=(0.25,0.25,0.25,0.25), smoothing_function=chencherry.method7)

    meteor_scores = [meteor_score([r.split() for r in refs[i]], preds[i].split()) for i in img_ids]
    results['METEOR'] = np.mean(meteor_scores)

    # -----------------------------
    # 2. CIDEr & SPICE
    # -----------------------------
    print("2. Computing CIDEr (and SPICE)...")
    gts = {imgid: refs[imgid] for imgid in img_ids}       # references (list of strings)
    res = {imgid: [preds[imgid]] for imgid in img_ids}    # predictions (list of single strings)

    # CIDEr
    cider_scorer = Cider()
    cider_score, _ = cider_scorer.compute_score(gts, res)
    results['CIDEr'] = cider_score
    
    # SPICE (Commented out due to previous CalledProcessError)
    # try:
    #     spice_scorer = Spice()
    #     spice_score, _ = spice_scorer.compute_score(gts, res)
    #     results['SPICE'] = spice_score
    # except Exception as e:
    #     print(f"Warning: SPICE failed (likely Java path/memory issue). Error: {e}")
    #     results['SPICE'] = None


    # -----------------------------
    # 3. N-gram diversity
    # -----------------------------
    print("3. Computing Diversity Metrics (N-gram Diversity)...")
    
    def ngram_diversity(captions, n=n_gram):
        all_ngrams = []
        for c in captions:
            all_ngrams.extend(list(ngrams(c.split(), n)))
        total = len(all_ngrams)
        unique = len(set(all_ngrams))
        return unique / total if total > 0 else 0

    results[f'{n_gram}-gram diversity'] = ngram_diversity(captions_list, n_gram)
    
    # # mBLEU@4
    # print("3b. Computing mBLEU@4 (Diversity - Inter-Caption Similarity)...")
    # def compute_mbleu(captions):
    #     scores = []
    #     for i, c1 in enumerate(captions):
    #         others = captions[:i] + captions[i+1:]
    #         if not others: continue
    #         # Use weights=(0,0,0,1) for BLEU@4 as done in original script
    #         bleu_scores = [sentence_bleu([o.split()], c1.split(), weights=(0,0,0,1)) for o in others]
    #         scores.append(np.mean(bleu_scores))
    #     return np.mean(scores) if scores else 0
    # results['mBLEU@4'] = compute_mbleu(captions_list)


    # -----------------------------
    # 4. Semantic alignment proxy (SBERT, batched)
    # -----------------------------
    print("4. Computing Semantic Alignment Proxy (SBERT Embeddings)...")
    
    # Use the pre-loaded global SBERT_MODEL
    sbert_model = SBERT_MODEL
    
    try:
        # Encode all predicted captions at once
        pred_embs = sbert_model.encode(captions_list, batch_size=64, convert_to_numpy=True, show_progress_bar=False)

        # Encode all reference captions at once and map back to image IDs
        all_ref_caps = [cap for caps in refs.values() for cap in caps]
        all_ref_embs = sbert_model.encode(all_ref_caps, batch_size=64, convert_to_numpy=True, show_progress_bar=False)

        ref_embs_dict = {}
        i = 0
        for imgid, caps in refs.items():
            ref_embs_dict[imgid] = all_ref_embs[i:i+len(caps)]
            i += len(caps)

        # Compute cosine similarity for each image
        semantic_sims = []
        for idx, imgid in enumerate(img_ids):
            pred_emb = pred_embs[idx]
            ref_embs = ref_embs_dict[imgid]
            # Normalize and compute cosine similarity
            pred_emb_norm = pred_emb / np.linalg.norm(pred_emb)
            sim = np.mean([np.dot(pred_emb_norm, re/np.linalg.norm(re)) for re in ref_embs])
            semantic_sims.append(sim)

        results['Semantic alignment proxy'] = np.mean(semantic_sims)
    except Exception as e:
        print(f"Warning: SBERT computation failed. Error: {e}")
        results['Semantic alignment proxy'] = None
        
    print("\n--- Evaluation Complete ---")
    return results

Skipping import of cpp extensions due to incompatible torch version 2.6.0+cu124 for torchao version 0.14.0         Please see GitHub issue #2919 for more info


# Human Metric Evaluation

- These metrics compare the captions to detailed human captions

#### Flickr30k Evaluation

In [9]:
flickr30k_captions_file_path = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\Flickr30k\dataset_flickr32k.json"
blip_flickr30k_captions_file_path = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\Flickr30k\blip_image_captioning_large_flickr30k_images_karpathy_split_captions.json"

blip_flickr30k_evaluation_results = evaluate_captioning_model(flickr30k_captions_file_path, blip_flickr30k_captions_file_path)

# -----------------------------
# Print Results
# -----------------------------
if blip_flickr30k_evaluation_results:
    print("\n--- Summary of Results ---")
    for metric, score in blip_flickr30k_evaluation_results.items():
        if score is not None:
            print(f"{metric:<25}: {score:.4f}")
        else:
            print(f"{metric:<25}: N/A (Failed to compute)")

--- Starting Evaluation ---
Ref File: C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\Flickr30k\dataset_flickr32k.json
Caps File: C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\Flickr30k\blip_image_captioning_large_flickr30k_images_karpathy_split_captions.json

1. Computing Classical Metrics (BLEU, METEOR)...
2. Computing CIDEr (and SPICE)...
3. Computing Diversity Metrics (N-gram Diversity)...
3b. Computing mBLEU@4 (Diversity - Inter-Caption Similarity)...
4. Computing Semantic Alignment Proxy (SBERT Embeddings)...

--- Evaluation Complete ---

--- Summary of Results ---
BLEU@1                   : 0.5634
BLEU@2                   : 0.3434
BLEU@3                   : 0.2042
BLEU@4                   : 0.1197
METEOR                   : 0.1812
CIDEr                    : 0.0249
4-gram diversity         : 0.6338
mBLEU@4                  : 0.0756
Semantic alignme

In [7]:
flickr30k_captions_file_path = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\Flickr30k\dataset_flickr32k.json"
florence2_flickr30k_captions_file_path = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\Flickr30k\florence_2_base_ft_flickr30k_images_karpathy_split_long_captions.json"

florence2_flickr30k_evaluation_results = evaluate_captioning_model(flickr30k_captions_file_path, florence2_flickr30k_captions_file_path)

# -----------------------------
# Print Results
# -----------------------------
if florence2_flickr30k_evaluation_results:
    print("\n--- Summary of Results ---")
    for metric, score in florence2_flickr30k_evaluation_results.items():
        if score is not None:
            print(f"{metric:<25}: {score:.4f}")
        else:
            print(f"{metric:<25}: N/A (Failed to compute)")

--- Starting Evaluation ---
Ref File: C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\Flickr30k\dataset_flickr32k.json
Caps File: C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\Flickr30k\florence_2_base_ft_flickr30k_images_karpathy_split_long_captions.json

1. Computing Classical Metrics (BLEU, METEOR)...
2. Computing CIDEr (and SPICE)...
3. Computing Diversity Metrics (N-gram Diversity)...
4. Computing Semantic Alignment Proxy (SBERT Embeddings)...

--- Evaluation Complete ---

--- Summary of Results ---
BLEU@1                   : 0.4298
BLEU@2                   : 0.2528
BLEU@3                   : 0.1476
BLEU@4                   : 0.0855
METEOR                   : 0.1834
CIDEr                    : 0.0011
4-gram diversity         : 0.4504
Semantic alignment proxy : 0.1194


In [11]:
flickr30k_captions_file_path = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\Flickr30k\dataset_flickr32k.json"
kosmos2_flickr30k_captions_file_path = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\Flickr30k\kosmos_2_patch14_224_flickr30k_images_karpathy_split_long_captions.json"

kosmos2_flickr30k_evaluation_results = evaluate_captioning_model(flickr30k_captions_file_path, kosmos2_flickr30k_captions_file_path)

# -----------------------------
# Print Results
# -----------------------------
if kosmos2_flickr30k_evaluation_results:
    print("\n--- Summary of Results ---")
    for metric, score in kosmos2_flickr30k_evaluation_results.items():
        if score is not None:
            print(f"{metric:<25}: {score:.4f}")
        else:
            print(f"{metric:<25}: N/A (Failed to compute)")

--- Starting Evaluation ---
Ref File: C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\Flickr30k\dataset_flickr32k.json
Caps File: C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\Flickr30k\kosmos_2_patch14_224_flickr30k_images_karpathy_split_long_captions.json

1. Computing Classical Metrics (BLEU, METEOR)...
2. Computing CIDEr (and SPICE)...
3. Computing Diversity Metrics (N-gram Diversity)...
3b. Computing mBLEU@4 (Diversity - Inter-Caption Similarity)...
4. Computing Semantic Alignment Proxy (SBERT Embeddings)...

--- Evaluation Complete ---

--- Summary of Results ---
BLEU@1                   : 0.4690
BLEU@2                   : 0.2826
BLEU@3                   : 0.1684
BLEU@4                   : 0.0992
METEOR                   : 0.2044
CIDEr                    : 0.0051
4-gram diversity         : 0.4492
mBLEU@4                  : 0.0751
Semantic alignment

In [12]:
flickr30k_captions_file_path = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\Flickr30k\dataset_flickr32k.json"
vit_gpt2_image_captioning_flickr30k_captions_file_path = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\Flickr30k\vit_gpt2_image_captioning_flickr30k_images_karpathy_split_captions.json"

vit_gpt2_flickr30k_evaluation_results = evaluate_captioning_model(flickr30k_captions_file_path, vit_gpt2_image_captioning_flickr30k_captions_file_path)

# -----------------------------
# Print Results
# -----------------------------
if vit_gpt2_flickr30k_evaluation_results:
    print("\n--- Summary of Results ---")
    for metric, score in vit_gpt2_flickr30k_evaluation_results.items():
        if score is not None:
            print(f"{metric:<25}: {score:.4f}")
        else:
            print(f"{metric:<25}: N/A (Failed to compute)")

--- Starting Evaluation ---
Ref File: C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\Flickr30k\dataset_flickr32k.json
Caps File: C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\Flickr30k\vit_gpt2_image_captioning_flickr30k_images_karpathy_split_captions.json

1. Computing Classical Metrics (BLEU, METEOR)...
2. Computing CIDEr (and SPICE)...
3. Computing Diversity Metrics (N-gram Diversity)...
3b. Computing mBLEU@4 (Diversity - Inter-Caption Similarity)...


c:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\.venv-Copy-Copy\lib\site-packages\nltk\translate\bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 2-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
c:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\.venv-Copy-Copy\lib\site-packages\nltk\translate\bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)


4. Computing Semantic Alignment Proxy (SBERT Embeddings)...

--- Evaluation Complete ---

--- Summary of Results ---
BLEU@1                   : 0.5984
BLEU@2                   : 0.3671
BLEU@3                   : 0.2205
BLEU@4                   : 0.1306
METEOR                   : 0.1890
CIDEr                    : 0.0306
4-gram diversity         : 0.4935
mBLEU@4                  : 0.0072
Semantic alignment proxy : 0.0682


In [13]:
flickr30k_captions_file_path = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\Flickr30k\dataset_flickr32k.json"
tiny_image_captioning_flickr30k_captions_file_path = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\Flickr30k\tiny_image_captioning_flickr30k_images_karpathy_split_captions.json"

tiny_image_captioning_flickr30k_evaluation_results = evaluate_captioning_model(flickr30k_captions_file_path, tiny_image_captioning_flickr30k_captions_file_path)

# -----------------------------
# Print Results
# -----------------------------
if tiny_image_captioning_flickr30k_evaluation_results:
    print("\n--- Summary of Results ---")
    for metric, score in tiny_image_captioning_flickr30k_evaluation_results.items():
        if score is not None:
            print(f"{metric:<25}: {score:.4f}")
        else:
            print(f"{metric:<25}: N/A (Failed to compute)")

--- Starting Evaluation ---
Ref File: C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\Flickr30k\dataset_flickr32k.json
Caps File: C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\Flickr30k\tiny_image_captioning_flickr30k_images_karpathy_split_captions.json

1. Computing Classical Metrics (BLEU, METEOR)...
2. Computing CIDEr (and SPICE)...
3. Computing Diversity Metrics (N-gram Diversity)...
3b. Computing mBLEU@4 (Diversity - Inter-Caption Similarity)...
4. Computing Semantic Alignment Proxy (SBERT Embeddings)...

--- Evaluation Complete ---

--- Summary of Results ---
BLEU@1                   : 0.6081
BLEU@2                   : 0.3785
BLEU@3                   : 0.2311
BLEU@4                   : 0.1386
METEOR                   : 0.1915
CIDEr                    : 0.0375
4-gram diversity         : 0.1499
mBLEU@4                  : 0.0557
Semantic alignment pro

#### COCO Evaluation

In [4]:
coco_captions_file_path = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\COCO2014\dataset_coco.json"
blip_coco2014_captions_file_path = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\COCO2014\blip_image_captioning_large_coco2014_karpathy_split_captions.json"

blip_coco2014_evaluation_results = evaluate_captioning_model(coco_captions_file_path, blip_coco2014_captions_file_path)

# -----------------------------
# Print Results
# -----------------------------
if blip_coco2014_evaluation_results:
    print("\n--- Summary of Results ---")
    for metric, score in blip_coco2014_evaluation_results.items():
        if score is not None:
            print(f"{metric:<25}: {score:.4f}")
        else:
            print(f"{metric:<25}: N/A (Failed to compute)")

--- Starting Evaluation ---
Ref File: C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\COCO2014\dataset_coco.json
Caps File: C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\COCO2014\blip_image_captioning_large_coco2014_karpathy_split_captions.json

1. Computing Classical Metrics (BLEU, METEOR)...
2. Computing CIDEr (and SPICE)...
3. Computing Diversity Metrics (N-gram Diversity)...
4. Computing Semantic Alignment Proxy (SBERT Embeddings)...

--- Evaluation Complete ---

--- Summary of Results ---
BLEU@1                   : 0.5649
BLEU@2                   : 0.3483
BLEU@3                   : 0.2092
BLEU@4                   : 0.1239
METEOR                   : 0.2017
CIDEr                    : 0.0313
4-gram diversity         : 0.5423
Semantic alignment proxy : 0.1165


In [8]:
coco_captions_file_path = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\COCO2014\dataset_coco.json"
florence2_coco2014_captions_file_path = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\COCO2014\florence_2_base_ft_coco2014_karpathy_split_long_captions.json"

florence2_coco2014_evaluation_results = evaluate_captioning_model(coco_captions_file_path, florence2_coco2014_captions_file_path)

# -----------------------------
# Print Results
# -----------------------------
if florence2_coco2014_evaluation_results:
    print("\n--- Summary of Results ---")
    for metric, score in florence2_coco2014_evaluation_results.items():
        if score is not None:
            print(f"{metric:<25}: {score:.4f}")
        else:
            print(f"{metric:<25}: N/A (Failed to compute)")

--- Starting Evaluation ---
Ref File: C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\COCO2014\dataset_coco.json
Caps File: C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\COCO2014\florence_2_base_ft_coco2014_karpathy_split_long_captions.json

1. Computing Classical Metrics (BLEU, METEOR)...
2. Computing CIDEr (and SPICE)...
3. Computing Diversity Metrics (N-gram Diversity)...
4. Computing Semantic Alignment Proxy (SBERT Embeddings)...

--- Evaluation Complete ---

--- Summary of Results ---
BLEU@1                   : 0.4252
BLEU@2                   : 0.2496
BLEU@3                   : 0.1455
BLEU@4                   : 0.0842
METEOR                   : 0.1754
CIDEr                    : 0.0013
4-gram diversity         : 0.2978
Semantic alignment proxy : 0.1373


In [ ]:
coco_captions_file_path = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\COCO2014\dataset_coco.json"
kosmos2_coco2014_captions_file_path = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\COCO2014\kosmos_2_patch14_224_coco2014_images_karpathy_split_long_captions.json"

kosmos2_coco2014_evaluation_results = evaluate_captioning_model(coco_captions_file_path, kosmos2_coco2014_captions_file_path)

# -----------------------------
# Print Results
# -----------------------------
if kosmos2_coco2014_evaluation_results:
    print("\n--- Summary of Results ---")
    for metric, score in kosmos2_coco2014_evaluation_results.items():
        if score is not None:
            print(f"{metric:<25}: {score:.4f}")
        else:
            print(f"{metric:<25}: N/A (Failed to compute)")

In [5]:
coco_captions_file_path = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\COCO2014\dataset_coco.json"
vit_gpt2_image_captioning_coco2014_captions_file_path = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\COCO2014\vit_gpt2_image_captioning_coco2014_karpathy_split_captions.json"

vit_gpt2_coco2014_evaluation_results = evaluate_captioning_model(coco_captions_file_path, vit_gpt2_image_captioning_coco2014_captions_file_path)

# -----------------------------
# Print Results
# -----------------------------
if vit_gpt2_coco2014_evaluation_results:
    print("\n--- Summary of Results ---")
    for metric, score in vit_gpt2_coco2014_evaluation_results.items():
        if score is not None:
            print(f"{metric:<25}: {score:.4f}")
        else:
            print(f"{metric:<25}: N/A (Failed to compute)")

--- Starting Evaluation ---
Ref File: C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\COCO2014\dataset_coco.json
Caps File: C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\COCO2014\vit_gpt2_image_captioning_coco2014_karpathy_split_captions.json

1. Computing Classical Metrics (BLEU, METEOR)...
2. Computing CIDEr (and SPICE)...
3. Computing Diversity Metrics (N-gram Diversity)...
4. Computing Semantic Alignment Proxy (SBERT Embeddings)...

--- Evaluation Complete ---

--- Summary of Results ---
BLEU@1                   : 0.5797
BLEU@2                   : 0.3566
BLEU@3                   : 0.2157
BLEU@4                   : 0.1290
METEOR                   : 0.1918
CIDEr                    : 0.0307
4-gram diversity         : 0.3246
Semantic alignment proxy : 0.0904


In [6]:
coco_captions_file_path = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\COCO2014\dataset_coco.json"
tiny_image_captioning_coco_captions_file_path = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\COCO2014\tiny_image_captioning_coco2014_karpathy_split_captions.json"

evaluation_results = evaluate_captioning_model(coco_captions_file_path, tiny_image_captioning_coco_captions_file_path)

# -----------------------------
# Print Results
# -----------------------------
if evaluation_results:
    print("\n--- Summary of Results ---")
    for metric, score in evaluation_results.items():
        if score is not None:
            print(f"{metric:<25}: {score:.4f}")
        else:
            print(f"{metric:<25}: N/A (Failed to compute)")

--- Starting Evaluation ---
Ref File: C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\COCO2014\dataset_coco.json
Caps File: C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\COCO2014\tiny_image_captioning_coco2014_karpathy_split_captions.json

1. Computing Classical Metrics (BLEU, METEOR)...
2. Computing CIDEr (and SPICE)...
3. Computing Diversity Metrics (N-gram Diversity)...
4. Computing Semantic Alignment Proxy (SBERT Embeddings)...

--- Evaluation Complete ---

--- Summary of Results ---
BLEU@1                   : 0.5697
BLEU@2                   : 0.3523
BLEU@3                   : 0.2132
BLEU@4                   : 0.1270
METEOR                   : 0.1964
CIDEr                    : 0.0367
4-gram diversity         : 0.0547
Semantic alignment proxy : 0.0562


# CLIP Similarity Test

- Aligned with the ['A Reference-free Evaluation Metric for Image Captioning'](https://aclanthology.org/2021.emnlp-main.595v2.pdf) paper this test uses CLIPScore & RefCLIPScore to detemine if the captioning models' captions fit the images.

In [ ]:
import os
import json
import torch
import pandas as pd
from tqdm import tqdm
from PIL import Image
from transformers import AutoProcessor, AutoModel

CLIP_MODEL_NAME = "openai/clip-vit-large-patch14"
BATCH_SIZE = 50

def _normalize_data(data):
    """
    Normalizes COCO/Flickr30k-style JSON structure into
    a flat list of dicts with keys: filename, caption, split.
    Example input format:
        { "dataset": "...", "images": [{ "filename": ..., "split": ..., "sentences": [{ "raw": ...}, ...]}]}
    """
    normalized = []
    if not isinstance(data, dict) or "images" not in data:
        return normalized

    for img_entry in data["images"]:
        filename = img_entry.get("filename")
        split = img_entry.get("split", "")
        sentences = img_entry.get("sentences", [])
        for s in sentences:
            caption = s.get("raw")
            if caption:
                normalized.append({
                    "filename": filename,
                    "caption": caption,
                    "split": split
                })
    return normalized

def harmonic_mean(a, b, eps=1e-8):
    return 2 * a * b / (a + b + eps)

def compute_all_clip_metrics(
    json_filepath_ref: str,
    json_filepath_gen: str,
    images_base_dir: str,
    split_filter_ref: str = None,
    split_filter_gen: str = None
) -> pd.DataFrame:
    """
    Unified function computing:
    Image-caption similarity (CLIP-S)
    Caption-caption similarity (cross-file)
    RefCLIP-S (harmonic mean of both)

    Includes raw cosine similarities for each component.

    Args:
        json_filepath_ref (str): JSON with reference captions.
        json_filepath_gen (str): JSON with generated captions.
        images_base_dir (str): Directory containing images.
        split_filter_ref / gen (str, optional): Optional split filters.

    Returns:
        pd.DataFrame: Metrics per generated caption including raw cosine similarities.
    """

    print(f"--- Starting Unified CLIP Metric Computation ---")
    print(f"Model: {CLIP_MODEL_NAME}")
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # Load CLIP model
    try:
        processor = AutoProcessor.from_pretrained(CLIP_MODEL_NAME, use_fast=True)
        model = AutoModel.from_pretrained(CLIP_MODEL_NAME).to(device)
    except Exception as e:
        print(f"❌ Error loading CLIP model: {e}")
        return pd.DataFrame()

    # Helper: load and group captions by filename
    def load_json_grouped(path, split_filter=None):
        with open(path, 'r', encoding='utf-8') as f:
            data = _normalize_data(json.load(f))
        if split_filter:
            data = [d for d in data if d.get("split", "").lower() == split_filter.lower()]
        grouped = {}
        for d in data:
            grouped.setdefault(d["filename"], []).append(d["caption"])
        return grouped

    ref_groups = load_json_grouped(json_filepath_ref, split_filter_ref)
    gen_groups = load_json_grouped(json_filepath_gen, split_filter_gen)
    common_images = set(ref_groups.keys()) & set(gen_groups.keys())
    print(f"Processing {len(common_images)} common images...")

    results = []

    for filename in tqdm(common_images, desc="Computing Metrics"):
        ref_captions = ref_groups[filename]
        gen_captions = gen_groups[filename]
        if not ref_captions or not gen_captions:
            continue

        # Load image
        img_path = os.path.join(images_base_dir, filename)
        try:
            image = Image.open(img_path).convert("RGB")
        except FileNotFoundError:
            tqdm.write(f"Image not found: {img_path}")
            continue

        # Image embedding
        image_inputs = processor(images=image, return_tensors="pt").to(device)
        with torch.no_grad():
            image_features = model.get_image_features(**image_inputs)
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)

        # Text embeddings
        all_captions = ref_captions + gen_captions
        inputs = processor(text=all_captions, return_tensors="pt", padding=True, truncation=True).to(device)
        with torch.no_grad():
            text_features = model.get_text_features(**inputs)
        text_features = text_features / text_features.norm(dim=-1, keepdim=True)

        # Split embeddings
        N_ref = len(ref_captions)
        ref_embeds = text_features[:N_ref]
        gen_embeds = text_features[N_ref:]

        # Compute metrics
        for gen_embed, gen_caption in zip(gen_embeds, gen_captions):
            # Raw cosine similarity: image-caption
            img_caption_cos = torch.sum(gen_embed * image_features).item()

            # CLIP-S
            clip_s = 2.5 * max(img_caption_cos, 0)

            # Caption–reference cosine similarities (vector)
            ref_sims_vec = torch.matmul(ref_embeds, gen_embed.unsqueeze(1)).squeeze(1).cpu().numpy()
            ref_sim = max(ref_sims_vec.max(), 0)

            # RefCLIP-S
            refclip_s = harmonic_mean(clip_s, ref_sim)

            results.append({
                "image_name": filename,
                "generated_caption": gen_caption,
                "CLIP_Score": clip_s,
                "CLIP_cosine": img_caption_cos,             # raw image-caption cosine
                "RefSim": ref_sim,
                "RefSim_vector": ref_sims_vec.tolist(),     # all reference cosines
                "RefCLIP_Score": refclip_s,
                "caption_count_ref": N_ref,
                "caption_count_gen": len(gen_captions)
            })

    df = pd.DataFrame(results)
    print(f"\n Computation Complete — Processed {len(df)} captions")
    if not df.empty:
        print(f"Average Image-Caption Cosine Similarity: {df['CLIP_cosine'].mean():.4f}")
        print(f"Average CLIP-S: {df['CLIP_Score'].mean():.4f}")
        print(f"Average RefSim: {df['RefSim'].mean():.4f}")
        print(f"Average RefCLIP-S: {df['RefCLIP_Score'].mean():.4f}")

    return df

## Flickr30k

In [133]:
flickr30k_image_dir = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\Flickr30k\flickr30k-images_karpathy_split" 
flickr30k_json_filepath = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\Flickr30k\dataset_flickr32k.json"

flickr30k_results = compute_all_clip_metrics(
    json_filepath_ref = flickr30k_json_filepath,
    json_filepath_gen = flickr30k_json_filepath,
    images_base_dir = flickr30k_image_dir,
    split_filter_ref = 'test',
    split_filter_gen = 'test'
)

--- Starting Unified CLIP Metric Computation ---
Model: openai/clip-vit-large-patch14
Processing 1000 common images...


Computing Metrics: 100%|██████████| 1000/1000 [02:31<00:00,  6.59it/s]


 Computation Complete — Processed 5000 captions
Average Image-Caption Cosine Similarity: 0.2686
Average CLIP-S: 0.6716
Average RefSim: 1.0000
Average RefCLIP-S: 0.7992


In [ ]:
flickr30k_image_dir = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\Flickr30k\flickr30k-images_karpathy_split" 
flickr30k_json_filepath = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\Flickr30k\dataset_flickr32k.json"
blip_flickr30k_json_filepath = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\Flickr30k\blip_image_captioning_large_flickr30k_images_karpathy_split_captions.json"

blip_flickr30k_results = compute_all_clip_metrics(
    json_filepath_ref = flickr30k_json_filepath,
    json_filepath_gen = blip_flickr30k_json_filepath,
    images_base_dir = flickr30k_image_dir,
    split_filter_ref = 'test',
    split_filter_gen = None
)

--- Starting Unified CLIP Metric Computation ---
Model: openai/clip-vit-large-patch14
Processing 1000 common images...


Computing Metrics: 100%|██████████| 1000/1000 [00:46<00:00, 21.46it/s]


 Computation Complete — Processed 1000 captions
Average Image-Caption Cosine Similarity: 0.2586
Average CLIP-S: 0.6466
Average RefSim: 0.7145
Average RefCLIP-S: 0.6741


In [103]:
flickr30k_image_dir = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\Flickr30k\flickr30k-images_karpathy_split" 
flickr30k_json_filepath = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\Flickr30k\dataset_flickr32k.json"
florence2_flickr30k_json_filepath = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\Flickr30k\florence_2_base_ft_flickr30k_images_karpathy_split_long_captions.json"

florence2_flickr30k_results = compute_all_clip_metrics(
    json_filepath_ref = flickr30k_json_filepath,
    json_filepath_gen = florence2_flickr30k_json_filepath,
    images_base_dir = flickr30k_image_dir,
    split_filter_ref = 'test',
    split_filter_gen = None
)

--- Starting Unified CLIP Metric Computation ---
Model: openai/clip-vit-large-patch14
Processing 1000 common images...


Computing Metrics: 100%|██████████| 1000/1000 [00:49<00:00, 20.15it/s]


 Computation Complete — Processed 1000 captions
Average Image-Caption Cosine Similarity: 0.2424
Average CLIP-S: 0.6059
Average RefSim: 0.5759
Average RefCLIP-S: 0.5853


In [104]:
flickr30k_image_dir = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\Flickr30k\flickr30k-images_karpathy_split" 
flickr30k_json_filepath = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\Flickr30k\dataset_flickr32k.json"
kosmos2_flickr30k_json_filepath = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\Flickr30k\kosmos_2_patch14_224_flickr30k_images_karpathy_split_long_captions.json"

kosmos2_flickr30k_results = compute_all_clip_metrics(
    json_filepath_ref = flickr30k_json_filepath,
    json_filepath_gen = kosmos2_flickr30k_json_filepath,
    images_base_dir = flickr30k_image_dir,
    split_filter_ref = 'test',
    split_filter_gen = None
)

--- Starting Unified CLIP Metric Computation ---
Model: openai/clip-vit-large-patch14
Processing 1000 common images...


Computing Metrics: 100%|██████████| 1000/1000 [00:50<00:00, 19.95it/s]


 Computation Complete — Processed 1000 captions
Average Image-Caption Cosine Similarity: 0.2240
Average CLIP-S: 0.5600
Average RefSim: 0.5766
Average RefCLIP-S: 0.5619


In [102]:
flickr30k_image_dir = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\Flickr30k\flickr30k-images_karpathy_split" 
flickr30k_json_filepath = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\Flickr30k\dataset_flickr32k.json"
tiny_flickr30k_json_filepath = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\Flickr30k\tiny_image_captioning_flickr30k_images_karpathy_split_captions.json"

tiny_flickr30k_results = compute_all_clip_metrics(
    json_filepath_ref = flickr30k_json_filepath,
    json_filepath_gen = tiny_flickr30k_json_filepath,
    images_base_dir = flickr30k_image_dir,
    split_filter_ref = 'test',
    split_filter_gen = None
)

--- Starting Unified CLIP Metric Computation ---
Model: openai/clip-vit-large-patch14
Processing 1000 common images...


Computing Metrics: 100%|██████████| 1000/1000 [00:48<00:00, 20.73it/s]


 Computation Complete — Processed 1000 captions
Average Image-Caption Cosine Similarity: 0.1911
Average CLIP-S: 0.4777
Average RefSim: 0.6371
Average RefCLIP-S: 0.5409


In [105]:
flickr30k_image_dir = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\Flickr30k\flickr30k-images_karpathy_split" 
flickr30k_json_filepath = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\Flickr30k\dataset_flickr32k.json"
vitgpt2_flickr30k_json_filepath = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\Flickr30k\vit_gpt2_image_captioning_flickr30k_images_karpathy_split_captions.json"

vitgpt2_flickr30k_results = compute_all_clip_metrics(
    json_filepath_ref = flickr30k_json_filepath,
    json_filepath_gen = vitgpt2_flickr30k_json_filepath,
    images_base_dir = flickr30k_image_dir,
    split_filter_ref = 'test',
    split_filter_gen = None
)

--- Starting Unified CLIP Metric Computation ---
Model: openai/clip-vit-large-patch14
Processing 1000 common images...


Computing Metrics: 100%|██████████| 1000/1000 [00:49<00:00, 20.38it/s]


 Computation Complete — Processed 1000 captions
Average Image-Caption Cosine Similarity: 0.2116
Average CLIP-S: 0.5290
Average RefSim: 0.6713
Average RefCLIP-S: 0.5867


## COCO2014

In [143]:
coco2014_image_dir = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\COCO2014\val2014_karpathy_split" 
coco2014_json_filepath = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\COCO2014\dataset_coco.json"

coco2014_results = compute_all_clip_metrics(
    json_filepath_ref = coco2014_json_filepath,
    json_filepath_gen = coco2014_json_filepath,
    images_base_dir = coco2014_image_dir,
    split_filter_ref = 'test',
    split_filter_gen = 'test'
)

--- Starting Unified CLIP Metric Computation ---
Model: openai/clip-vit-large-patch14
Processing 5000 common images...


Computing Metrics: 100%|██████████| 5000/5000 [23:18<00:00,  3.58it/s]  



 Computation Complete — Processed 25010 captions
Average Image-Caption Cosine Similarity: 0.2573
Average CLIP-S: 0.6431
Average RefSim: 1.0000
Average RefCLIP-S: 0.7786


In [106]:
coco2014_image_dir = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\COCO2014\val2014_karpathy_split" 
coco2014_json_filepath = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\COCO2014\dataset_coco.json"
blip_coco2014_json_filepath = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\COCO2014\blip_image_captioning_large_coco2014_karpathy_split_captions.json"

blip_coco2014_results = compute_all_clip_metrics(
    json_filepath_ref = coco2014_json_filepath,
    json_filepath_gen = blip_coco2014_json_filepath,
    images_base_dir = coco2014_image_dir,
    split_filter_ref = 'test',
    split_filter_gen = None
)

--- Starting Unified CLIP Metric Computation ---
Model: openai/clip-vit-large-patch14


'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 3cf1ae31-157a-4972-ab3b-24d0319e0b3d)')' thrown while requesting HEAD https://huggingface.co/openai/clip-vit-large-patch14/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].


Processing 5000 common images...


Computing Metrics: 100%|██████████| 5000/5000 [04:13<00:00, 19.74it/s]


 Computation Complete — Processed 5000 captions
Average Image-Caption Cosine Similarity: 0.2626
Average CLIP-S: 0.6566
Average RefSim: 0.7858
Average RefCLIP-S: 0.7110


In [107]:
coco2014_image_dir = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\COCO2014\val2014_karpathy_split" 
coco2014_json_filepath = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\COCO2014\dataset_coco.json"
florence2_coco2014_json_filepath = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\COCO2014\florence_2_base_ft_coco2014_karpathy_split_long_captions.json"

florence2_coco2014_results = compute_all_clip_metrics(
    json_filepath_ref = coco2014_json_filepath,
    json_filepath_gen = florence2_coco2014_json_filepath,
    images_base_dir = coco2014_image_dir,
    split_filter_ref = 'test',
    split_filter_gen = None
)

--- Starting Unified CLIP Metric Computation ---
Model: openai/clip-vit-large-patch14
Processing 5000 common images...


Computing Metrics: 100%|██████████| 5000/5000 [04:18<00:00, 19.35it/s]


 Computation Complete — Processed 5000 captions
Average Image-Caption Cosine Similarity: 0.2371
Average CLIP-S: 0.5927
Average RefSim: 0.6536
Average RefCLIP-S: 0.6163


In [108]:
coco2014_image_dir = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\COCO2014\val2014_karpathy_split" 
coco2014_json_filepath = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\COCO2014\dataset_coco.json"
kosmos2_coco2014_json_filepath = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\COCO2014\kosmos_2_patch14_224_coco2014_karpathy_split_long_captions.json"

kosmos2_coco2014_results = compute_all_clip_metrics(
    json_filepath_ref = coco2014_json_filepath,
    json_filepath_gen = kosmos2_coco2014_json_filepath,
    images_base_dir = coco2014_image_dir,
    split_filter_ref = 'test',
    split_filter_gen = None
)

--- Starting Unified CLIP Metric Computation ---
Model: openai/clip-vit-large-patch14
Processing 5000 common images...


Computing Metrics: 100%|██████████| 5000/5000 [04:17<00:00, 19.43it/s]


 Computation Complete — Processed 5000 captions
Average Image-Caption Cosine Similarity: 0.2227
Average CLIP-S: 0.5568
Average RefSim: 0.6314
Average RefCLIP-S: 0.5861


In [109]:
coco2014_image_dir = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\COCO2014\val2014_karpathy_split" 
coco2014_json_filepath = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\COCO2014\dataset_coco.json"
tiny_coco2014_json_filepath = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\COCO2014\tiny_image_captioning_coco2014_karpathy_split_captions.json"

tiny_coco2014_results = compute_all_clip_metrics(
    json_filepath_ref = coco2014_json_filepath,
    json_filepath_gen = tiny_coco2014_json_filepath,
    images_base_dir = coco2014_image_dir,
    split_filter_ref = 'test',
    split_filter_gen = None
)

--- Starting Unified CLIP Metric Computation ---
Model: openai/clip-vit-large-patch14
Processing 5000 common images...


Computing Metrics: 100%|██████████| 5000/5000 [04:13<00:00, 19.75it/s]


 Computation Complete — Processed 5000 captions
Average Image-Caption Cosine Similarity: 0.1644
Average CLIP-S: 0.4110
Average RefSim: 0.6087
Average RefCLIP-S: 0.4853


In [110]:
coco2014_image_dir = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\COCO2014\val2014_karpathy_split" 
coco2014_json_filepath = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\COCO2014\dataset_coco.json"
vitgpt2_coco2014_json_filepath = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\COCO2014\vit_gpt2_image_captioning_coco2014_karpathy_split_captions.json"

vitgpt2_coco2014_results = compute_all_clip_metrics(
    json_filepath_ref = coco2014_json_filepath,
    json_filepath_gen = vitgpt2_coco2014_json_filepath,
    images_base_dir = coco2014_image_dir,
    split_filter_ref = 'test',
    split_filter_gen = None
)

--- Starting Unified CLIP Metric Computation ---
Model: openai/clip-vit-large-patch14
Processing 5000 common images...


Computing Metrics: 100%|██████████| 5000/5000 [04:14<00:00, 19.67it/s]


 Computation Complete — Processed 5000 captions
Average Image-Caption Cosine Similarity: 0.2416
Average CLIP-S: 0.6040
Average RefSim: 0.8289
Average RefCLIP-S: 0.6951


## Tabulated Results

- Clip Similarity Score of **3.0** is considered a good heuristic for estimating semantic image-text-content matching as per [LAION](https://laion.ai/blog/laion-400-open-dataset/#:~:text=The%20threshold%20of%200.3%20had%20been%20determined%20through%20human%20evaluations%20and%20seemed%20to%20be%20a%20good%20heuristic%20for%20estimating%20semantic%20image%2Dtext%2Dcontent%20matching.)

In [125]:
def display_consolidated_summary(summary_entries: List[Dict[str, Any]]):
    """
    Generates and displays a consolidated summary of average CLIP-based metrics
    from multiple evaluation runs, including:
        - Image-Caption Cosine Similarity
        - CLIP-S
        - RefSim
        - RefCLIP-S

    Args:
        summary_entries (List[Dict[str, Any]]): Each dict should include:
            - 'df': pandas DataFrame returned from compute_all_clip_metrics
            - 'model_name': Name of the evaluated model
            - 'dataset_info': Dataset or split information
    """
    final_summary_data = []

    for entry in summary_entries:
        df = entry.get('df')
        model_name = entry.get('model_name', 'N/A Model')
        dataset_info = entry.get('dataset_info', 'N/A Dataset')

        if df is None or df.empty:
            print(f"⚠️ Skipping summary for '{model_name} / {dataset_info}' - DataFrame is empty.")
            continue

        # Compute averages for all metrics if they exist
        avg_img_caption_cos = df['CLIP_cosine'].mean() if 'CLIP_cosine' in df.columns else float('nan')
        avg_clip_s = df['CLIP_Score'].mean() if 'CLIP_Score' in df.columns else float('nan')
        avg_refsim = df['RefSim'].mean() if 'RefSim' in df.columns else float('nan')
        avg_refclip_s = df['RefCLIP_Score'].mean() if 'RefCLIP_Score' in df.columns else float('nan')
        # avg_cross_caption = df['avg_cross_file_caption_similarity'].mean() if 'avg_cross_file_caption_similarity' in df.columns else float('nan')

        final_summary_data.append({
            'Model Name': model_name,
            'Dataset/Split': dataset_info,
            'Avg Image-Caption Cosine': avg_img_caption_cos,
            'Avg CLIP-S': avg_clip_s,
            'Avg RefSim': avg_refsim,
            'Avg RefCLIP-S': avg_refclip_s,
            # 'Avg Cross-Caption': avg_cross_caption
        })

    print("\n" + "="*100)
    print("--- CONSOLIDATED CLIP METRICS SUMMARY TABLE ---")
    print("="*100)

    if final_summary_data:
        summary_df = pd.DataFrame(final_summary_data)
        # Sort by RefCLIP-S descending as a default
        summary_df = summary_df.sort_values(by='Avg RefCLIP-S', ascending=False)
        print(summary_df.to_string(index=False, float_format="%.4f"))
    else:
        print("No valid evaluation results were generated to summarize.")
    print("="*100)

    # Metric description
    print("\nMetric Descriptions:")
    print("1. Avg Image-Caption Cosine:")
    print("   - Raw cosine similarity between generated caption embedding and image embedding.")
    print("   - Range: [-1, 1]; higher is better (closer alignment).")
    print("2. Avg CLIP-S:")
    print("   - Scaled image-caption similarity: 2.5 * max(cosine, 0).")
    print("   - Range: [0, 2.5]; higher indicates better alignment between image and caption.")
    print("3. Avg RefSim:")
    print("   - Maximum cosine similarity between a generated caption and all reference captions for the same image.")
    print("   - Range: [0, 1]; higher indicates closer match to references.")
    print("4. Avg RefCLIP-S:")
    print("   - Harmonic mean of CLIP-S and RefSim (paper definition).")
    print("   - Range: [0, 2.5]; combines image alignment and reference caption similarity.")
    # print("5. Avg Cross-Caption:")
    # print("   - Mean cosine similarity between each generated caption and all reference captions (average pairwise).")
    # print("   - Range: [-1, 1]; higher indicates higher semantic similarity to references.\n")

In [145]:
coco2014_results_to_summarize = [
    {
        'df': coco2014_results,
        'model_name': 'N/A',
        'dataset_info': "Karpathy COCO2014 (Test Split)"

    },
    {
        'df': blip_coco2014_results, 
        'model_name': "BLIP Large", 
        'dataset_info': "Karpathy COCO2014 (Test Split)"
    },
    {
        'df': florence2_coco2014_results, 
        'model_name': "Florence 2 Base", 
        'dataset_info': "Karpathy COCO2014 (Test Split)"
    },
    {
        'df': kosmos2_coco2014_results, 
        'model_name': "Kosmos 2 Patch14 224", 
        'dataset_info': "Karpathy COCO2014 (Test Split)"
    },
    {
        'df': tiny_coco2014_results, 
        'model_name': "Tiny", 
        'dataset_info': "Karpathy COCO2014 (Test Split)"
    },
    {
        'df': vitgpt2_coco2014_results, 
        'model_name': "Vit GPT 2", 
        'dataset_info': "Karpathy COCO2014 (Test Split)"
    }
]
display_consolidated_summary(coco2014_results_to_summarize)


--- CONSOLIDATED CLIP METRICS SUMMARY TABLE ---
          Model Name                  Dataset/Split  Avg Image-Caption Cosine  Avg CLIP-S  Avg RefSim  Avg RefCLIP-S
                 N/A Karpathy COCO2014 (Test Split)                    0.2573      0.6431      1.0000         0.7786
          BLIP Large Karpathy COCO2014 (Test Split)                    0.2626      0.6566      0.7858         0.7110
           Vit GPT 2 Karpathy COCO2014 (Test Split)                    0.2416      0.6040      0.8289         0.6951
     Florence 2 Base Karpathy COCO2014 (Test Split)                    0.2371      0.5927      0.6536         0.6163
Kosmos 2 Patch14 224 Karpathy COCO2014 (Test Split)                    0.2227      0.5568      0.6314         0.5861
                Tiny Karpathy COCO2014 (Test Split)                    0.1644      0.4110      0.6087         0.4853

Metric Descriptions:
1. Avg Image-Caption Cosine:
   - Raw cosine similarity between generated caption embedding and image embeddin

In [146]:
flickr30k_results_to_summarize = [
    {
        'df': flickr30k_results,
        'model_name': 'N/A',
        'dataset_info':  "Karpathy Flickr30k (Test Split)"

    },
    {
        'df': blip_flickr30k_results, 
        'model_name': "BLIP Large", 
        'dataset_info': "Karpathy Flickr30k (Test Split)"
    },
    {
        'df': florence2_flickr30k_results, 
        'model_name': "Florence 2 Base", 
        'dataset_info': "Karpathy Flickr30k (Test Split)"
    },
    {
        'df': kosmos2_flickr30k_results, 
        'model_name': "Kosmos 2 Patch14 224", 
        'dataset_info': "Karpathy Flickr30k (Test Split)"
    },
    {
        'df': tiny_flickr30k_results, 
        'model_name': "Tiny", 
        'dataset_info': "Karpathy Flickr30k (Test Split)"
    },
    {
        'df': vitgpt2_flickr30k_results, 
        'model_name': "Vit GPT 2", 
        'dataset_info': "Karpathy Flickr30k (Test Split)"
    }
]
display_consolidated_summary(flickr30k_results_to_summarize)


--- CONSOLIDATED CLIP METRICS SUMMARY TABLE ---
          Model Name                   Dataset/Split  Avg Image-Caption Cosine  Avg CLIP-S  Avg RefSim  Avg RefCLIP-S
                 N/A Karpathy Flickr30k (Test Split)                    0.2686      0.6716      1.0000         0.7992
          BLIP Large Karpathy Flickr30k (Test Split)                    0.2586      0.6466      0.7145         0.6741
           Vit GPT 2 Karpathy Flickr30k (Test Split)                    0.2116      0.5290      0.6713         0.5867
     Florence 2 Base Karpathy Flickr30k (Test Split)                    0.2424      0.6059      0.5759         0.5853
Kosmos 2 Patch14 224 Karpathy Flickr30k (Test Split)                    0.2240      0.5600      0.5766         0.5619
                Tiny Karpathy Flickr30k (Test Split)                    0.1911      0.4777      0.6371         0.5409

Metric Descriptions:
1. Avg Image-Caption Cosine:
   - Raw cosine similarity between generated caption embedding and image e

## Exection Speed 

- Testing the 5 viable Models on LAION-5B 10k Subset

### Constructing the LAION-5B-10k Image Set

In [ ]:
import os
import shutil
import random
from tqdm import tqdm
from PIL import Image, UnidentifiedImageError

def copy_limited_files(source_dir, dest_dir, limit=100):
    """
    Randomly copies up to `limit` valid image files from `source_dir` to `dest_dir`.
    Skips corrupted or invalid images (width/height <= 1).
    Shows a tqdm progress bar.

    Parameters:
        source_dir (str): Path to source directory.
        dest_dir (str): Path to destination directory.
        limit (int): Maximum number of valid files to copy.
    """

    os.makedirs(dest_dir, exist_ok=True)

    # List all files in source directory (not directories)
    all_files = [f for f in os.listdir(source_dir) if os.path.isfile(os.path.join(source_dir, f))]
    random.shuffle(all_files)

    copied = 0
    total_checked = 0
    skipped = 0

    with tqdm(total=limit, desc="Copying valid images", unit="file") as pbar:
        for filename in all_files:
            if copied >= limit:
                break

            src_path = os.path.join(source_dir, filename)
            dest_path = os.path.join(dest_dir, filename)

            try:
                with Image.open(src_path) as img:
                    w, h = img.size
                    if w <= 1 or h <= 1:
                        skipped += 1
                        continue

                shutil.copy2(src_path, dest_path)
                copied += 1
                pbar.update(1)

            except (UnidentifiedImageError, OSError):
                skipped += 1
                continue

            total_checked += 1

    print(f"\n✅ Copied {copied} valid images to {dest_dir}")
    print(f"⚠️ Skipped {skipped} corrupted/invalid images")

# Example usage
source = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\test_image_cache\part-00000-checkpoint-1426200-old"
dest = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\LAION-5B-10k\LAION-5B-10k-images-2"
limit = 10_000

# copy_limited_files(source, dest, limit)

BLIP

In [ ]:
"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\.venv-Copy-Copy\Scripts\python.exe" "C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioning\blip-image-captioning-large.py" --image_dir "C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\LAION-5B-10k\LAION-5B-10k-images" --batch_size 8 --num_workers 8 --output_file "C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\LAION-5B-10k\blip_image_captioning_large_laion5b10k.json" --prompt "An image of"
# 18 Mins 38 Secs

Florence 2

In [ ]:
"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\.venv-Copy-Copy\Scripts\python.exe" "C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioning\florence-2-base-ft.py" --image_dir "C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\LAION-5B-10k\LAION-5B-10k-images" --batch_size 8 --num_workers 8 --output_file "C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\LAION-5B-10k\florence_2_base_ft_laion5b10k.json" --prompt "<DETAILED_CAPTION>"
# 23 Mins 37 Secs

Kosmos 2

In [ ]:
"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\.venv-Copy-Copy\Scripts\python.exe" "C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioning\kosmos-2-patch14-224.py" --image_dir "C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\LAION-5B-10k\LAION-5B-10k-images" --batch_size 8 --num_workers 8 --output_file "C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\LAION-5B-10k\kosmos_2_patch14_224_laion5b10k.json" --prompt "<grounding> Describe this image in detail:"
# 

Vit GPT 2

In [ ]:
"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\.venv-Copy-Copy\Scripts\python.exe" "C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioning\vit-gpt2-image-captioning.py" --image_dir "C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\LAION-5B-10k\LAION-5B-10k-images" --batch_size 8 --num_workers 8 --output_file "C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\LAION-5B-10k\vit_gpt2_image_captioning_laion5b10k.json"
# 11 Mins 24 Secs

Tiny

In [ ]:
"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\.venv-Copy-Copy\Scripts\python.exe" "C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioning\tiny-image-captioning.py" --image_dir "C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\LAION-5B-10k\LAION-5B-10k-images" --batch_size 8 --num_workers 8 --output_file "C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\LAION-5B-10k\tiny_image_captioning_laion5b10k.json"
# 17 Mins 54 Secs

In [152]:
from PIL import Image, UnidentifiedImageError
import os

from PIL import Image, UnidentifiedImageError
import os

def validate_images_strict(image_dir, extensions=(".jpg", ".jpeg", ".png")):
    """
    Validates all images in a directory.
    Detects images that open but are partially corrupted or have wrong mode/shape.

    Returns:
        bad_images (list): list of problematic file paths
    """
    bad_images = []
    total = 0

    for filename in os.listdir(image_dir):
        if not filename.lower().endswith(extensions):
            continue

        total += 1
        path = os.path.join(image_dir, filename)

        try:
            with Image.open(path) as img:
                # Try to load the full image into memory
                img.load()

                # Force conversion to RGB (some grayscale or RGBA cause model errors)
                img = img.convert("RGB")

                # Sanity check: ensure valid dimensions
                if img.width < 10 or img.height < 10:
                    raise ValueError(f"Invalid dimensions: {img.size}")

        except (UnidentifiedImageError, OSError, ValueError) as e:
            print(f"⚠️ Bad image detected: {filename} — {e}")
            bad_images.append(path)

    print(f"\n✅ Scan complete: {total} images checked, {len(bad_images)} problematic.\n")
    return bad_images


validate_images_strict(r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\Spurious_Feature\ImageCaptioningEvaluationDatasets\LAION-5B-10k\LAION-5B-10k-images")

⚠️ Bad image detected: 12402.jpg — Invalid dimensions: (1, 1)
⚠️ Bad image detected: 12740.jpg — Invalid dimensions: (1, 1)
⚠️ Bad image detected: 1491.jpg — Invalid dimensions: (1, 1)
⚠️ Bad image detected: 17343.jpg — Invalid dimensions: (1, 1)
⚠️ Bad image detected: 17914.jpg — Invalid dimensions: (1, 1)
⚠️ Bad image detected: 17957.jpg — Invalid dimensions: (1, 1)
⚠️ Bad image detected: 18689.jpg — Invalid dimensions: (1, 1)
⚠️ Bad image detected: 20704.jpg — Invalid dimensions: (1, 1)
⚠️ Bad image detected: 21252.jpg — Invalid dimensions: (1, 1)
⚠️ Bad image detected: 22038.jpg — Invalid dimensions: (1, 1)
⚠️ Bad image detected: 22914.jpg — Invalid dimensions: (1, 1)
⚠️ Bad image detected: 23718.jpg — Invalid dimensions: (1, 1)
⚠️ Bad image detected: 3144.jpg — Invalid dimensions: (1, 1)
⚠️ Bad image detected: 4065.jpg — Invalid dimensions: (1, 1)
⚠️ Bad image detected: 485.jpg — Invalid dimensions: (1, 1)
⚠️ Bad image detected: 6707.jpg — Invalid dimensions: (1, 1)
⚠️ Bad image d

['C:\\MastersRepos\\ARI5902-Research-Topics-in-AI\\LAION-5B Testing\\Spurious_Feature\\ImageCaptioningEvaluationDatasets\\LAION-5B-10k\\LAION-5B-10k-images\\12402.jpg',
 'C:\\MastersRepos\\ARI5902-Research-Topics-in-AI\\LAION-5B Testing\\Spurious_Feature\\ImageCaptioningEvaluationDatasets\\LAION-5B-10k\\LAION-5B-10k-images\\12740.jpg',
 'C:\\MastersRepos\\ARI5902-Research-Topics-in-AI\\LAION-5B Testing\\Spurious_Feature\\ImageCaptioningEvaluationDatasets\\LAION-5B-10k\\LAION-5B-10k-images\\1491.jpg',
 'C:\\MastersRepos\\ARI5902-Research-Topics-in-AI\\LAION-5B Testing\\Spurious_Feature\\ImageCaptioningEvaluationDatasets\\LAION-5B-10k\\LAION-5B-10k-images\\17343.jpg',
 'C:\\MastersRepos\\ARI5902-Research-Topics-in-AI\\LAION-5B Testing\\Spurious_Feature\\ImageCaptioningEvaluationDatasets\\LAION-5B-10k\\LAION-5B-10k-images\\17914.jpg',
 'C:\\MastersRepos\\ARI5902-Research-Topics-in-AI\\LAION-5B Testing\\Spurious_Feature\\ImageCaptioningEvaluationDatasets\\LAION-5B-10k\\LAION-5B-10k-images\

# Conclusion

Based on the evaluation conducted above the **BLIP** Model was determined to be the best performing image captioning model.